# GSE149383 — Drug-Tolerance Trajectory Analysis (PC9 + Erlotinib)

**This notebook is organised to match the manuscript one-to-one.** Every code
cell is labelled with the manuscript section it implements and the exact
number, table, or figure it produces, in the order those items appear in the
paper. Run top to bottom; nothing is computed out of order and nothing is
computed twice.

| Manuscript section | What it covers | Code section below |
|---|---|---|
| §2.1 | Data source, download, cohort assembly | 1 |
| §2.2 | Quality control | 2 |
| §2.3–2.4 | Normalisation, HVG selection, PCA | 3 |
| §2.5 | Depth-confound regression | 4 → **Figure 1** |
| §2.6 | Diffusion pseudotime & distance-from-baseline | 5 → **Figures 2–3** |
| §2.7 | Correlation dimension (main trajectory) | 6 |
| §2.11 | Depth-matching & reversibility correlation dimension | 7 → **Table 1, Figure 4** |
| — | Reversibility UMAP | 8 → **Figure 6** |
| §2.9 | Cross-validated, permutation-tested classification | 9 |
| §2.8 | Marker-free trajectory axis (circularity safeguard) | 10 |
| §2.8 | CCM with surrogate testing | 11 → **Table 2, Figures 7–8** |
| §2.10 | Marker gene validation | 12 → **Figures 9–10** |
| §3.7 | Sensitivity of the CCM attribution to the QC threshold | 13 → **Figures 5, 11** |
| §2.9 (Bandt & Pompe) | Permutation entropy & shuffle-surrogate test | 14 → **Table 6, Figure 12** |
| §2.9 (Schreiber) | Transfer entropy | 15 → **Table 7, Figure 13** |
| §3.8 (Discussion) | Eckmann–Ruelle data-sufficiency check | 16 |

**Note on figure numbers below.** Every figure produced by this notebook is
labelled and saved under the SAME number it has in the manuscript (Figure 1
through Figure 13, all 13 present) — not the order the code sections happen
to run in. Figure 5 (correlation dimension, original vs. alternative QC
threshold) and Figure 11 (CCM sensitivity to QC threshold) can only be built
once code section 13 has run both thresholds, so they appear later in the
notebook than they do in the manuscript; the filename and in-figure title of
every saved PNG still say exactly which manuscript figure it is.

**Runtime note (Colab):** the first cell installs pinned package versions and
restarts the runtime once. This is a one-time step — after it reconnects,
choose *Runtime → Run all* and everything below runs in a clean environment
without asking again.

## Setup — package install (Colab only, one-time restart) & imports

In [ ]:
# One-time setup & install. The runtime restarts once after this cell; this
# is expected, not a crash. After it reconnects, Runtime -> Run all continues
# past this cell automatically (it detects the marker file and skips ahead).
import os, subprocess, sys

SETUP_MARKER = "/content/.gse149383_setup_done"

if not os.path.exists(SETUP_MARKER):
    print("Installing packages (first run only)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                     "--force-reinstall", "numpy==1.26.4", "scipy==1.13.1"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                     "anndata==0.10.9", "scanpy==1.10.3",
                     "skccm", "seaborn", "scikit-learn",
                     "diptest", "statsmodels"], check=True)
    with open(SETUP_MARKER, "w") as f:
        f.write("done")
    print("Setup complete. Restarting the runtime now (this is required).")
    print("After it reconnects, choose Runtime -> Run all to continue.")
    os.kill(os.getpid(), 9)
else:
    print("Setup already completed in this session/runtime -- skipping install.")

In [ ]:
# Imports used throughout. Corresponds to the software listed in Methods §2.12.
import os, gzip, subprocess, warnings
import numpy as np, pandas as pd, scipy.sparse as sp, anndata as ad, scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.spatial.distance import pdist
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, permutation_test_score
from skccm import Embed, CCM
from skccm.utilities import train_test_split as ccm_train_test_split

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

os.makedirs('/content/gse149383', exist_ok=True)
os.chdir('/content/gse149383')

DT_MARKERS_WANTED = ['EGFR', 'TACSTD2', 'SERPINE1', 'CD24', 'MET', 'ALDH1A3', 'SOX4', 'GPRC5A', 'CYP1B1']
CONDITION_ORDER = ['D0', 'D1', 'D2', 'D4', 'D9', 'D11_pooled', 'D19_DMSO_holiday', 'D19_ERL_retreated']
MAIN_ARM = ['D0', 'D1', 'D2', 'D4', 'D9', 'D11_pooled']
DAY_MAP = {'D0': 0, 'D1': 1, 'D2': 2, 'D4': 4, 'D9': 9, 'D11_pooled': 11}

print("Imports OK. numpy", np.__version__, "| scipy", __import__('scipy').__version__)

## Code section 1 — §2.1: data source, download, cohort assembly

File URLs were confirmed by parsing the Series' own SOFT metadata directly,
rather than assumed from GSM titles — this is how the two duplicate samples
(D0_rep2, D2_rep2) were caught before they could distort any result. Per-
sample DGE files omit genes with zero counts entirely rather than writing an
explicit zero, so a union of gene lists across all nine files is built first,
and any gene missing from a given sample is filled with zero for that
sample's cells.

In [ ]:
# §2.1 — Download the nine real, non-duplicate sample files.
gsm_to_url = {
    "GSM3972657": ("D0",  "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3972nnn/GSM3972657/suppl/GSM3972657_D0.dge.txt.gz"),
    "GSM3972658": ("D1",  "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3972nnn/GSM3972658/suppl/GSM3972658_D1.dge.txt.gz"),
    "GSM3972659": ("D2",  "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3972nnn/GSM3972659/suppl/GSM3972659_D2.dge.txt.gz"),
    "GSM3972660": ("D4",  "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3972nnn/GSM3972660/suppl/GSM3972660_D4.dge.txt.gz"),
    "GSM3972661": ("D9",  "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3972nnn/GSM3972661/suppl/GSM3972661_D9.dge.txt.gz"),
    "GSM3972662": ("D11", "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3972nnn/GSM3972662/suppl/GSM3972662_D11.dge.txt.gz"),
    "GSM3972671": ("D11_rep2", "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3972nnn/GSM3972671/suppl/GSM3972671_D11_ERL.dge.txt.gz"),
    "GSM3972672": ("D19_DMSO_holiday", "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3972nnn/GSM3972672/suppl/GSM3972672_D19DMSO.dge.txt.gz"),
    "GSM3972673": ("D19_ERL_retreated", "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3972nnn/GSM3972673/suppl/GSM3972673_D19_ERL.dge.txt.gz"),
}
# GSM3972669 ("D0_rep2") and GSM3972670 ("D2_rep2") are deliberately excluded --
# confirmed via MD5 hash and cell-barcode-set comparison to be exact duplicates
# of GSM3972657 and GSM3972659 respectively, not independent biological replicates.

downloaded_files = {}
for gsm, (label, url) in gsm_to_url.items():
    fname = url.split('/')[-1]
    local_path = f"/content/gse149383/{fname}"
    if not os.path.exists(local_path):
        subprocess.run(["wget", "-q", "-O", local_path, url])
    downloaded_files[gsm] = {"path": local_path, "label": label}

print(f"Downloaded {len(downloaded_files)} real, non-duplicate samples")

In [ ]:
# §2.1 (continued) — Combine into one AnnData object (union of gene lists),
# and pool the two independent D11 depositions into a single 'D11_pooled'
# condition (confirmed comparable beforehand: R^2 < 0.05 on every leading PC).
gene_union = set()
for gsm, info in downloaded_files.items():
    with gzip.open(info["path"], 'rt') as f:
        f.readline()
        for line in f:
            gene_union.add(line.split('\t', 1)[0])

gene_list = sorted(gene_union)
gene_to_idx = {g: i for i, g in enumerate(gene_list)}

all_i, all_j, all_v = [], [], []
cell_names, cell_label = [], []
col_offset = 0

for gsm, info in downloaded_files.items():
    df = pd.read_csv(info["path"], sep='\t', index_col=0)
    local_to_global = np.array([gene_to_idx[g] for g in df.index])
    arr = df.values
    nz_r, nz_c = np.nonzero(arr)
    all_i.append(local_to_global[nz_r])
    all_j.append(nz_c + col_offset)
    all_v.append(arr[nz_r, nz_c])
    cell_names.extend([f"{gsm}_{bc}" for bc in df.columns])
    cell_label.extend([info["label"]] * df.shape[1])
    col_offset += df.shape[1]
    del df, arr

X = sp.csr_matrix(
    (np.concatenate(all_v).astype(np.float32), (np.concatenate(all_j), np.concatenate(all_i))),
    shape=(col_offset, len(gene_list))
)

adata_raw = ad.AnnData(X=X)
adata_raw.var_names = gene_list
adata_raw.obs_names = cell_names
adata_raw.obs['sample_label'] = cell_label

group_map = {
    'D0': 'D0', 'D1': 'D1', 'D2': 'D2', 'D4': 'D4', 'D9': 'D9',
    'D11': 'D11_pooled', 'D11_rep2': 'D11_pooled',
    'D19_DMSO_holiday': 'D19_DMSO_holiday', 'D19_ERL_retreated': 'D19_ERL_retreated'
}
adata_raw.obs['condition'] = adata_raw.obs['sample_label'].map(group_map)
adata_raw.obs['condition'] = pd.Categorical(adata_raw.obs['condition'], categories=CONDITION_ORDER, ordered=True)

print(adata_raw)

## Code section 2 — §2.2 / §3.1: quality control

QC is applied to raw counts, before normalisation. The threshold (>=300 total
counts, >=30 detected genes, <30% mitochondrial fraction) was chosen after a
per-condition barcode-rank ("knee") check. Running this cell should reproduce
the cell counts reported in Table 1 of the manuscript.

**Update, addressing expert feedback:** QC metrics (total counts, genes
detected, mitochondrial fraction) are now computed with scanpy's own
`sc.pp.calculate_qc_metrics()` rather than by hand. Previously this notebook
computed the same metrics manually in two separate places (here, and again
inside the QC-sensitivity function in section 3.7), which is the most likely
source of the cell-count discrepancy flagged during expert review of the analysis between this cell and
that function's "original threshold" branch. Both now call the identical
scanpy function, so a fresh run of this cell and of section 3.7 should agree
exactly when the same threshold is used. This notebook has not been re-executed
end to end since this fix, since the required GEO download is not available in
this environment; running it fresh and confirming the two branches now agree
is the next step before the affected tables and figures are finalised.


In [ ]:
# §2.2 -- Quality control (raw counts, before normalisation).
# Updated per expert feedback: QC metrics are now
# computed with scanpy's own sc.pp.calculate_qc_metrics() rather than by
# hand. This cell and the reusable pipeline function in section 3.7 now both
# call the same scanpy function, so the "original" threshold branch inside
# that function is guaranteed to reproduce this cell's cell counts exactly,
# rather than risk silent drift between two independently hand-written QC
# implementations, which is what we believe caused the reproducibility
# discrepancy flagged during expert review of the analysis.
adata_raw.var['mt'] = adata_raw.var_names.str.upper().str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata_raw, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

# Aliased to the column names used throughout the rest of this notebook.
adata_raw.obs['n_genes_detected'] = adata_raw.obs['n_genes_by_counts']
adata_raw.obs['mito_fraction'] = adata_raw.obs['pct_counts_mt'] / 100.0  # scanpy reports a percentage, not a fraction

qc_mask = (
    (adata_raw.obs['total_counts'] >= 300)
    & (adata_raw.obs['n_genes_detected'] >= 30)
    & (adata_raw.obs['mito_fraction'] < 0.3)
).values
adata_qc2 = adata_raw[qc_mask].copy()
adata_qc2.layers['raw_counts'] = adata_qc2.X.copy()

print(f"{qc_mask.sum()} / {len(qc_mask)} cells passed QC   <-- §3.1 total")
print(adata_qc2.obs['condition'].value_counts().reindex(CONDITION_ORDER))


## Code section 3 — §2.3–2.4: normalisation, HVG selection, PCA (uncorrected)

Log1p(CPM/10k) normalisation, top-2000 dispersion-ranked genes plus the nine
literature markers force-included, then an *uncorrected* PCA — deliberately
computed here, before depth regression, so it can be compared side by side
with the corrected PCA in the next section for Figure 1.

In [ ]:
# §2.3-2.4 -- Normalise and select highly-variable genes.
sc.pp.normalize_total(adata_qc2, target_sum=1e4)
sc.pp.log1p(adata_qc2)
sc.pp.highly_variable_genes(adata_qc2, n_top_genes=2000, flavor='seurat')

dt_markers = [g for g in DT_MARKERS_WANTED if g in adata_qc2.var_names]
for gene in dt_markers:
    adata_qc2.var.loc[gene, 'highly_variable'] = True

adata_hvg2 = adata_qc2[:, adata_qc2.var['highly_variable']].copy()
adata_hvg2.obs['log_total_counts'] = np.log1p(adata_qc2.obs['total_counts'].values)
adata_hvg2.obs['mito_fraction'] = adata_qc2.obs['mito_fraction'].values

# Uncorrected PCA -- same HVG set, scaled, but WITHOUT regressing out depth.
# Kept only for the "before" panel of Figure 1; all downstream analysis (from
# section 4 onward) uses the depth-corrected embedding built in the next cell.
adata_before = adata_hvg2.copy()
sc.pp.scale(adata_before, max_value=10)
sc.tl.pca(adata_before, n_comps=50, svd_solver='arpack')

print(f"{len(dt_markers)} of 9 literature markers found in the data:", dt_markers)

## Code section 4 — §2.5: regress out sequencing depth → Figure 1

Depth is a *continuous* covariate here, not nested inside condition, so it is
removed by per-gene multiple linear regression (`scanpy.pp.regress_out`)
rather than a categorical batch-correction tool such as Harmony. This cell
reproduces the r = 0.864 → −0.003 (values reported in the manuscript) result reported in §3.1 and renders
**Figure 1**.

In [ ]:
# §2.5 -- Regress out sequencing depth as a continuous covariate, then rebuild
# the PCA. This corrected embedding (adata_hvg2.obsm['X_pca']) is what every
# later section (trajectory, correlation dimension, classification, CCM) uses.
sc.pp.regress_out(adata_hvg2, ['log_total_counts', 'mito_fraction'])
sc.pp.scale(adata_hvg2, max_value=10)
sc.tl.pca(adata_hvg2, n_comps=50, svd_solver='arpack')

log_depth_all = adata_hvg2.obs['log_total_counts'].values
pc1_before = adata_before.obsm['X_pca'][:, 0]
pc1_after = adata_hvg2.obsm['X_pca'][:, 0]
r_before = np.corrcoef(pc1_before, log_depth_all)[0, 1]
r_after = np.corrcoef(pc1_after, log_depth_all)[0, 1]
print(f"PC1-vs-depth correlation: before={r_before:.3f}, after={r_after:.3f}   <-- §3.1")

In [ ]:
# Figure 1 -- PCA before vs. after depth correction, coloured by log-depth.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sca = axes[0].scatter(adata_before.obsm['X_pca'][:, 0], adata_before.obsm['X_pca'][:, 1],
                       c=log_depth_all, cmap='viridis', s=4)
axes[0].set_title(f'Before correction (PC1 vs. depth r={r_before:.3f})')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
plt.colorbar(sca, ax=axes[0], label='log total counts')

scb = axes[1].scatter(adata_hvg2.obsm['X_pca'][:, 0], adata_hvg2.obsm['X_pca'][:, 1],
                       c=log_depth_all, cmap='viridis', s=4)
axes[1].set_title(f'After correction (PC1 vs. depth r={r_after:.3f})')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
plt.colorbar(scb, ax=axes[1], label='log total counts')

plt.suptitle('Figure 1. PCA before vs. after regressing out sequencing depth')
plt.tight_layout()
plt.savefig('figure1_pca_depth_correction.png', dpi=300, bbox_inches='tight')
plt.show()

## Code section 5 — §2.6: diffusion pseudotime & distance-from-baseline → Figures 2–3

Two independent trajectory measures are built from the depth-corrected PCA:
diffusion pseudotime (manifold-based) and Euclidean distance from the day-0
centroid (assumption-light). Reported in §3.2; rendered as **Figure 2**
(UMAP by day / by pseudotime) and **Figure 3** (both measures side by
side).

In [ ]:
# §2.6 -- Diffusion map, pseudotime, and the Euclidean distance-from-baseline axis.

# --- Environment fix (added post-submission) ---------------------------------
# sc.tl.umap() internally does `import umap`, and umap-learn's __init__.py
# unconditionally tries to import its optional ParametricUMAP extra, which
# needs tensorflow -> jax. In some Colab/Python 3.13 environments this chain
# fails with `AttributeError: module 'numpy.dtypes' has no attribute
# 'StringDType'` (a numpy/jax/tensorflow version mismatch), rather than the
# ImportError that umap's own try/except is built to catch, so the error
# propagates and kills the cell even though we never use ParametricUMAP.
# We don't use parametric (neural-net-based) UMAP anywhere in this notebook,
# only the standard embedding, so it is safe to make `import tensorflow`
# fail fast and let umap's own except-ImportError branch skip it cleanly.
import sys
if 'tensorflow' not in sys.modules:
    sys.modules['tensorflow'] = None
# -------------------------------------------------------------------------

sc.pp.neighbors(adata_hvg2, n_neighbors=15, use_rep='X_pca')
sc.tl.diffmap(adata_hvg2, n_comps=15)
sc.tl.umap(adata_hvg2)

d0_mask = (adata_hvg2.obs['condition'] == 'D0').values
d0_centroid = adata_hvg2.obsm['X_pca'][d0_mask].mean(axis=0)
root_idx = np.where(d0_mask)[0][np.argmin(np.linalg.norm(adata_hvg2.obsm['X_pca'][d0_mask] - d0_centroid, axis=1))]
adata_hvg2.uns['iroot'] = root_idx
sc.tl.dpt(adata_hvg2)

# Primary trajectory axis used throughout the paper: simpler than pseudotime,
# no manifold assumptions, and independently confirmed to show the same
# overshoot-then-partial-recovery shape (§3.2).
adata_hvg2.obs['dist_from_baseline'] = np.linalg.norm(adata_hvg2.obsm['X_pca'] - d0_centroid, axis=1)

print("Mean distance from D0 centroid, by day  <-- §3.2:")
print(adata_hvg2.obs.groupby('condition', observed=True)['dist_from_baseline'].mean().reindex(MAIN_ARM))
print("\nMean diffusion pseudotime, by day  <-- §3.2:")
print(adata_hvg2.obs.groupby('condition', observed=True)['dpt_pseudotime'].mean().reindex(MAIN_ARM))


In [ ]:
# Figure 2 -- UMAP of the main day 0 -> day 11 trajectory, by day and by pseudotime.
main_arm_mask_plot = adata_hvg2.obs['condition'].isin(MAIN_ARM).values
umap_main = adata_hvg2.obsm['X_umap'][main_arm_mask_plot]
day_numeric = adata_hvg2.obs['condition'].map(DAY_MAP).values[main_arm_mask_plot].astype(float)
dpt_main = adata_hvg2.obs['dpt_pseudotime'].values[main_arm_mask_plot]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sca = axes[0].scatter(umap_main[:, 0], umap_main[:, 1], c=day_numeric, cmap='viridis', s=4)
axes[0].set_title('UMAP coloured by treatment day')
axes[0].set_xlabel('UMAP1'); axes[0].set_ylabel('UMAP2')
plt.colorbar(sca, ax=axes[0], label='Day')

scb = axes[1].scatter(umap_main[:, 0], umap_main[:, 1], c=dpt_main, cmap='viridis', s=4)
axes[1].set_title('UMAP coloured by diffusion pseudotime')
axes[1].set_xlabel('UMAP1'); axes[1].set_ylabel('UMAP2')
plt.colorbar(scb, ax=axes[1], label='Pseudotime')

plt.suptitle('Figure 2. UMAP of the D0 -> D11 trajectory, by day and by pseudotime')
plt.tight_layout()
plt.savefig('figure2_umap_trajectory.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 3 -- distance-from-baseline and diffusion pseudotime, both axes, by day.
grp = adata_hvg2.obs[adata_hvg2.obs['condition'].isin(MAIN_ARM)].groupby('condition', observed=True)
dist_mean = grp['dist_from_baseline'].mean().reindex(MAIN_ARM)
dist_sem = grp['dist_from_baseline'].sem().reindex(MAIN_ARM)
dpt_mean = grp['dpt_pseudotime'].mean().reindex(MAIN_ARM)
dpt_sem = grp['dpt_pseudotime'].sem().reindex(MAIN_ARM)
x_pos = np.arange(len(MAIN_ARM))

fig, ax1 = plt.subplots(figsize=(7, 5))
ax1.errorbar(x_pos, dist_mean.values, yerr=dist_sem.values, marker='o', color='tab:blue',
             label='Distance from baseline (left axis)')
ax1.set_xlabel('Condition')
ax1.set_ylabel('Mean distance from day-0 centroid', color='tab:blue')
ax1.set_xticks(x_pos); ax1.set_xticklabels(MAIN_ARM)
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()
ax2.errorbar(x_pos, dpt_mean.values, yerr=dpt_sem.values, marker='s', linestyle='--', color='tab:red',
             label='Diffusion pseudotime (right axis)')
ax2.set_ylabel('Mean diffusion pseudotime', color='tab:red')
ax2.tick_params(axis='y', labelcolor='tab:red')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=8)
plt.title('Figure 3. Trajectory reconstruction: two independent measures')
plt.tight_layout()
plt.savefig('figure3_trajectory_dual_axis.png', dpi=300, bbox_inches='tight')
plt.show()

## Code section 6 — §2.7: correlation dimension (main trajectory)

The Grassberger–Procaccia estimator (Eq. 4–5 in the manuscript), applied on
the top 15 corrected PCs, with 200 cell-level bootstrap resamples for the
95% CI. Defined once here and reused, unchanged, for the reversibility pair
in the next section and for the sensitivity check in §3.7.

In [ ]:
# §2.7 -- Correlation dimension estimator + bootstrap CI. Reused everywhere below.
def correlation_dimension(points, n_radii=20, scaling_frac=(0.15, 0.6)):
    dists = pdist(points)
    dists = dists[dists > 0]
    r_min, r_max = np.percentile(dists, [5, 95])
    radii = np.logspace(np.log10(r_min), np.log10(r_max), n_radii)
    C_r = np.array([(dists < r).mean() for r in radii])
    valid = C_r > 0
    log_r, log_C = np.log(radii[valid]), np.log(C_r[valid])
    n = len(log_r)
    lo, hi = int(n * scaling_frac[0]), int(n * scaling_frac[1])
    if hi <= lo:
        hi = lo + 2
    slope, _ = np.polyfit(log_r[lo:hi], log_C[lo:hi], 1)
    return slope

def bootstrap_corr_dim(points, n_boot=200, sample_frac=0.8, seed=0):
    rng = np.random.default_rng(seed)
    n = len(points)
    sample_size = max(int(n * sample_frac), 15)
    out = []
    for _ in range(n_boot):
        idx = rng.choice(n, size=sample_size, replace=True)
        try:
            out.append(correlation_dimension(points[idx]))
        except Exception:
            continue
    return np.array(out)

corr_dim_results = {}
embedding = adata_hvg2.obsm['X_pca'][:, :15]
for cond in MAIN_ARM:
    mask = (adata_hvg2.obs['condition'] == cond).values
    boot = bootstrap_corr_dim(embedding[mask])
    corr_dim_results[cond] = (mask.sum(), boot.mean(), np.percentile(boot, 2.5), np.percentile(boot, 97.5))
    print(f"{cond}: n={mask.sum()}, corr_dim={boot.mean():.3f} "
          f"[{np.percentile(boot, 2.5):.3f}, {np.percentile(boot, 97.5):.3f}]   <-- Table 1")

## Code section 7 — §2.11: depth-matching, reversibility correlation dimension → Table 1, Figure 4

Cells from the two D19 conditions are binned into 10 joint depth-quantile
bins and downsampled to the smaller group's count within each bin, giving two
depth-comparable groups of 862 cells each. Correlation dimension is then
computed for both, using the same estimator as section 6, and everything is
assembled into **Table 1** and **Figure 4**.

In [ ]:
# §2.11 -- Depth-matching for the reversibility comparison.
def depth_match_two_groups(adata, cond_col, cond_a, cond_b, depth_col='total_counts', n_bins=10, seed=0):
    rng = np.random.default_rng(seed)
    mask_a = (adata.obs[cond_col] == cond_a).values
    mask_b = (adata.obs[cond_col] == cond_b).values
    depth_a = adata.obs.loc[mask_a, depth_col].values
    depth_b = adata.obs.loc[mask_b, depth_col].values
    combined = np.concatenate([depth_a, depth_b])
    edges = np.quantile(combined, np.linspace(0, 1, n_bins + 1))
    edges[-1] += 1
    idx_a, idx_b = np.where(mask_a)[0], np.where(mask_b)[0]
    keep_a, keep_b = [], []
    for i in range(n_bins):
        in_a = idx_a[(depth_a >= edges[i]) & (depth_a < edges[i + 1])]
        in_b = idx_b[(depth_b >= edges[i]) & (depth_b < edges[i + 1])]
        m = min(len(in_a), len(in_b))
        if m > 0:
            keep_a.extend(rng.choice(in_a, m, replace=False))
            keep_b.extend(rng.choice(in_b, m, replace=False))
    return np.array(keep_a), np.array(keep_b)

idx_dmso, idx_erl = depth_match_two_groups(adata_qc2, 'condition', 'D19_DMSO_holiday', 'D19_ERL_retreated')
matched_names = np.concatenate([adata_qc2.obs_names[idx_dmso], adata_qc2.obs_names[idx_erl]])
adata_reversibility = adata_hvg2[adata_hvg2.obs_names.isin(matched_names)].copy()

print(f"Depth-matched reversibility set: {adata_reversibility.n_obs} cells")
print(adata_reversibility.obs['condition'].value_counts())

embedding_rev = adata_reversibility.obsm['X_pca'][:, :15]
for cond in ['D19_DMSO_holiday', 'D19_ERL_retreated']:
    mask = (adata_reversibility.obs['condition'] == cond).values
    boot = bootstrap_corr_dim(embedding_rev[mask])
    corr_dim_results[cond] = (mask.sum(), boot.mean(), np.percentile(boot, 2.5), np.percentile(boot, 97.5))
    print(f"{cond}: n={mask.sum()}, corr_dim={boot.mean():.3f} "
          f"[{np.percentile(boot, 2.5):.3f}, {np.percentile(boot, 97.5):.3f}]   <-- Table 1")

In [ ]:
# Table 1 & Figure 4 -- correlation dimension, all eight conditions.
table1 = pd.DataFrame(
    {c: corr_dim_results[c] for c in CONDITION_ORDER},
    index=['n_cells', 'D2_mean', 'ci_lo', 'ci_hi']
).T
print("Table 1. Correlation dimension by condition:")
print(table1)

means = table1['D2_mean'].values
err_lo = means - table1['ci_lo'].values
err_hi = table1['ci_hi'].values - means
point_colors = ['tab:gray'] * 6 + ['tab:orange', 'tab:blue']

fig, ax = plt.subplots(figsize=(9, 5))
x_pos = np.arange(len(CONDITION_ORDER))
for i in range(len(CONDITION_ORDER)):
    ax.errorbar(x_pos[i], means[i], yerr=[[err_lo[i]], [err_hi[i]]], fmt='o', capsize=4, color=point_colors[i])
ax.axvline(5.5, linestyle=':', color='black')
ax.set_xticks(x_pos)
ax.set_xticklabels(CONDITION_ORDER, rotation=30, ha='right')
ax.set_ylabel('Correlation dimension (D2)')
ax.set_title('Figure 4. Correlation dimension by condition, 95% bootstrap CI')
plt.tight_layout()
plt.savefig('figure4_correlation_dimension.png', dpi=300, bbox_inches='tight')
plt.show()

## Code section 8 — Reversibility UMAP → Figure 6

In [ ]:
# Figure 6 -- UMAP of the depth-matched reversibility comparison, 862 cells/condition.
umap_rev = adata_reversibility.obsm['X_umap']
cond_rev = adata_reversibility.obs['condition'].values

fig, ax = plt.subplots(figsize=(6, 5))
for cond, color in zip(['D19_DMSO_holiday', 'D19_ERL_retreated'], ['tab:orange', 'tab:blue']):
    m = cond_rev == cond
    ax.scatter(umap_rev[m, 0], umap_rev[m, 1], s=6, alpha=0.6, label=cond, color=color)
ax.legend()
ax.set_xlabel('UMAP1'); ax.set_ylabel('UMAP2')
ax.set_title('Figure 6. Depth-matched reversibility comparison (862 cells/condition)')
plt.tight_layout()
plt.savefig('figure6_reversibility_umap.png', dpi=300, bbox_inches='tight')
plt.show()

## Code section 9 — §2.9: cross-validated, permutation-tested classification

5-fold stratified cross-validation on the top 20 corrected PCs, with a
200-shuffle permutation test for significance (§3.4).

In [ ]:
# §2.9 -- Logistic regression, 5-fold CV, 200-shuffle permutation test.
X_check = adata_reversibility.obsm['X_pca'][:, :20]
y_check = (adata_reversibility.obs['condition'] == 'D19_ERL_retreated').astype(int).values

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
clf = LogisticRegression(max_iter=1000)
cv_scores = cross_val_score(clf, X_check, y_check, cv=cv)
print(f"5-fold cross-validated accuracy: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}   <-- §3.4")

score, perm_scores, pvalue = permutation_test_score(clf, X_check, y_check, cv=cv, n_permutations=200, random_state=0)
print(f"Permutation test: accuracy={score:.3f}, p-value={pvalue:.4f}   <-- §3.4")

## Code section 10 — §2.8: marker-free trajectory axis (circularity safeguard)

Rebuilds the trajectory axis with the nine marker genes excluded from feature
selection entirely, so CCM can test each marker's causal role against an axis
that was not partly built from that same marker (§2.8's circularity
safeguard).

In [ ]:
# §2.8 -- Marker-free trajectory axis.
genes_for_basis = [g for g in adata_qc2.var_names if g not in dt_markers]
adata_basis = adata_qc2[adata_hvg2.obs_names, genes_for_basis].copy()
sc.pp.highly_variable_genes(adata_basis, n_top_genes=2000, flavor='seurat')
adata_basis_hvg = adata_basis[:, adata_basis.var['highly_variable']].copy()

adata_basis_hvg.obs['log_total_counts'] = np.log1p(
    adata_qc2.obs.loc[adata_basis_hvg.obs_names, 'total_counts'].values)
adata_basis_hvg.obs['mito_fraction'] = adata_qc2.obs.loc[adata_basis_hvg.obs_names, 'mito_fraction'].values

sc.pp.regress_out(adata_basis_hvg, ['log_total_counts', 'mito_fraction'])
sc.pp.scale(adata_basis_hvg, max_value=10)
sc.tl.pca(adata_basis_hvg, n_comps=50, svd_solver='arpack')

d0_mask_clean = adata_basis_hvg.obs_names.isin(
    adata_hvg2.obs_names[(adata_hvg2.obs['condition'] == 'D0').values])
centroid_clean = adata_basis_hvg.obsm['X_pca'][d0_mask_clean].mean(axis=0)
dist_clean_series = pd.Series(
    np.linalg.norm(adata_basis_hvg.obsm['X_pca'] - centroid_clean, axis=1),
    index=adata_basis_hvg.obs_names
)

axis_corr = np.corrcoef(adata_hvg2.obs['dist_from_baseline'].values,
                         dist_clean_series.loc[adata_hvg2.obs_names].values)[0, 1]
print("Correlation between original and marker-free axis:", axis_corr, "  <-- §2.8")

## Code section 11 — §2.8: CCM with surrogate testing → Table 2, Figures 7–8

Bins the marker-free trajectory axis and each marker gene's expression into
30 ordered bins, then runs convergent cross mapping bidirectionally between
each gene and the axis, testing significance against 200 randomly shuffled
surrogates (§3.5). The skill scores go into **Table 2** and **Figure 7**
(bar chart, both directions); the full skill-vs-library-length curves that
the surrogate test is actually run on go into **Figure 8**.

In [ ]:
# §2.8 -- Bin the trajectory and run CCM with surrogate significance testing.
main_arm_mask = adata_hvg2.obs['condition'].isin(MAIN_ARM).values
main_arm_names = adata_hvg2.obs_names[main_arm_mask]

real_day = adata_hvg2.obs.loc[main_arm_names, 'condition'].map(DAY_MAP).values
within_day_dist = dist_clean_series.loc[main_arm_names].values
order_key = real_day + (within_day_dist - within_day_dist.min()) / (within_day_dist.max() - within_day_dist.min() + 1e-9)
order_idx = np.argsort(order_key)
ordered_names = main_arm_names[order_idx]

n_bins = 30
bin_edges = np.linspace(order_key.min(), order_key.max(), n_bins + 1)
bin_idx = np.clip(np.digitize(order_key[order_idx], bin_edges) - 1, 0, n_bins - 1)

axis_binned = np.array([
    dist_clean_series.loc[ordered_names].values[bin_idx == b].mean() if (bin_idx == b).sum() > 0 else np.nan
    for b in range(n_bins)
])

gene_binned = {}
for gene in dt_markers:
    gidx = np.where(adata_qc2.var_names == gene)[0][0]
    expr = pd.Series(np.asarray(adata_qc2[:, gidx].X.todense()).flatten(), index=adata_qc2.obs_names)
    expr_ordered = expr.loc[ordered_names].values
    gene_binned[gene] = np.array([
        expr_ordered[bin_idx == b].mean() if (bin_idx == b).sum() > 0 else np.nan
        for b in range(n_bins)
    ])

binned_df = pd.DataFrame({**gene_binned, 'axis': axis_binned}).interpolate(limit_direction='both')

def run_ccm_pair_curve(x, y, embed_dim=2, lag=1, train_percent=0.75):
    e1, e2 = Embed(x), Embed(y)
    X1, X2 = e1.embed_vectors_1d(lag, embed_dim), e2.embed_vectors_1d(lag, embed_dim)
    x1tr, x1te, x2tr, x2te = ccm_train_test_split(X1, X2, percent=train_percent)
    ccm = CCM()
    ccm.fit(x1tr, x2tr)
    lib_lengths = np.arange(8, len(x1tr), max(1, (len(x1tr) - 8) // 6))
    ccm.predict(x1te, x2te, lib_lengths)
    sc1, sc2 = ccm.score()
    return lib_lengths, np.array(sc1), np.array(sc2)

def run_ccm_pair(x, y, **kwargs):
    _, sc1, sc2 = run_ccm_pair_curve(x, y, **kwargs)
    return sc1, sc2

def surrogate_test(x, y, n_surrogates=200, seed=0):
    rng = np.random.default_rng(seed)
    sc_xy, sc_yx = run_ccm_pair(x, y)
    real_xy, real_yx = np.nanmax(sc_xy), np.nanmax(sc_yx)
    surr_xy, surr_yx = [], []
    for _ in range(n_surrogates):
        try:
            sxy, syx = run_ccm_pair(x, rng.permutation(y))
            surr_xy.append(np.nanmax(sxy))
            surr_yx.append(np.nanmax(syx))
        except Exception:
            continue
    return real_xy, real_yx, (np.array(surr_xy) >= real_xy).mean(), (np.array(surr_yx) >= real_yx).mean()

table2_rows = []
for gene in dt_markers:
    real_xy, real_yx, p_xy, p_yx = surrogate_test(binned_df[gene].values, binned_df['axis'].values)
    table2_rows.append({'gene': gene, 'gene_to_axis_rho': real_xy, 'gene_to_axis_p': p_xy,
                         'axis_to_gene_rho': real_yx, 'axis_to_gene_p': p_yx})
    print(f"{gene}: gene->axis={real_xy:.3f} (p={p_xy:.3f}) axis->gene={real_yx:.3f} (p={p_yx:.3f})   <-- Table 2")

table2 = pd.DataFrame(table2_rows).set_index('gene')
print("\nTable 2. CCM results, marker-free trajectory axis:")
print(table2)

In [ ]:
# Figure 7 -- CCM skill scores for each marker gene against the marker-free
# trajectory axis, both directions (grouped bar chart). Asterisk = empirical
# p < 0.05 against 200 surrogates. Reuses table2 from the cell above.
genes_f7 = table2.index.tolist()
x_pos = np.arange(len(genes_f7))
width = 0.38

fig, ax = plt.subplots(figsize=(11, 4.5))
bars1 = ax.bar(x_pos - width / 2, table2['gene_to_axis_rho'], width,
                label='gene -> trajectory', color='tab:blue')
bars2 = ax.bar(x_pos + width / 2, table2['axis_to_gene_rho'], width,
                label='trajectory -> gene', color='tab:orange')

for i, gene in enumerate(genes_f7):
    if table2.loc[gene, 'gene_to_axis_p'] < 0.05:
        y = table2.loc[gene, 'gene_to_axis_rho']
        ax.text(x_pos[i] - width / 2, y + (0.03 if y >= 0 else -0.07), '*',
                ha='center', fontsize=13, color='tab:blue')
    if table2.loc[gene, 'axis_to_gene_p'] < 0.05:
        y = table2.loc[gene, 'axis_to_gene_rho']
        ax.text(x_pos[i] + width / 2, y + (0.03 if y >= 0 else -0.07), '*',
                ha='center', fontsize=13, color='tab:orange')

ax.axhline(0, color='gray', linewidth=0.7)
ax.set_xticks(x_pos)
ax.set_xticklabels(genes_f7, rotation=30, ha='right', fontstyle='italic')
ax.set_ylabel('CCM cross-map skill (rho)')
ax.set_title('Figure 7. Convergent cross mapping skill scores, both directions')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('figure7_ccm_skill_bars.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 8 -- CCM skill-vs-library-length curves for each marker gene, both
# directions. This is the actual convergence criterion the method is named
# for (rise-and-plateau shape), rather than the single endpoint value plotted
# in Figure 7 above.
n_genes = len(dt_markers)
fig, axes = plt.subplots(1, n_genes, figsize=(3 * n_genes, 3.2), sharey=True)
if n_genes == 1:
    axes = [axes]

for ax, gene in zip(axes, dt_markers):
    x = binned_df[gene].values
    y = binned_df['axis'].values
    lib_lengths, sc1, sc2 = run_ccm_pair_curve(x, y)
    p_xy, p_yx = table2.loc[gene, 'gene_to_axis_p'], table2.loc[gene, 'axis_to_gene_p']

    ax.plot(lib_lengths[:len(sc1)], sc1, marker='o', label='gene->axis', color='tab:blue')
    ax.plot(lib_lengths[:len(sc2)], sc2, marker='s', label='axis->gene', color='tab:red')

    title = gene
    if p_xy < 0.05:
        title += '*'
    if p_yx < 0.05:
        title += '*'
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Library length')

axes[0].set_ylabel('CCM skill (rho)')
axes[0].legend(fontsize=8)
plt.suptitle('Figure 8. CCM skill vs. library length (asterisk = empirical p<0.05 vs. 200 surrogates)')
plt.tight_layout()
plt.savefig('figure8_ccm_skill_curves.png', dpi=300, bbox_inches='tight')
plt.show()

## Code section 12 — §2.10 / §3.6: marker gene validation → Figures 9–10

Expression of all nine markers across the main trajectory (Figure 9), and
directly compared between drug-holiday and re-treatment with Mann–Whitney
p-values per gene (Figure 10).

In [ ]:
# Figure 9 -- marker expression across the main day 0 -> day 11 trajectory.
fig, axes = plt.subplots(2, 5, figsize=(20, 7))
axes = axes.flatten()
for i, gene in enumerate(dt_markers):
    gidx = np.where(adata_qc2.var_names == gene)[0][0]
    expr = np.asarray(adata_qc2[adata_hvg2.obs_names, gidx].X.todense()).flatten()
    df_plot = pd.DataFrame({'condition': adata_hvg2.obs['condition'].values, 'expr': expr})
    order_plot = MAIN_ARM
    sns.violinplot(data=df_plot[df_plot['condition'].isin(order_plot)], x='condition', y='expr',
                    order=order_plot, ax=axes[i], inner='quartile')
    axes[i].set_title(gene)
    axes[i].tick_params(axis='x', rotation=45)
for j in range(len(dt_markers), len(axes)):
    axes[j].axis('off')
plt.suptitle('Figure 9. Marker expression across the D0 -> D11 trajectory')
plt.tight_layout()
plt.savefig('figure9_markers_trajectory.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 10 -- the same nine markers, drug-holiday vs. re-treatment, with Mann-Whitney p-values.
fig2, axes2 = plt.subplots(2, 5, figsize=(20, 7))
axes2 = axes2.flatten()
for i, gene in enumerate(dt_markers):
    gidx = np.where(adata_qc2.var_names == gene)[0][0]
    expr = np.asarray(adata_qc2[adata_reversibility.obs_names, gidx].X.todense()).flatten()
    df_plot = pd.DataFrame({'condition': adata_reversibility.obs['condition'].values, 'expr': expr})
    sns.violinplot(data=df_plot, x='condition', y='expr', ax=axes2[i], inner='quartile')
    axes2[i].set_title(gene)
    axes2[i].tick_params(axis='x', rotation=30)
    stat, pval = stats.mannwhitneyu(
        df_plot.loc[df_plot['condition'] == 'D19_DMSO_holiday', 'expr'],
        df_plot.loc[df_plot['condition'] == 'D19_ERL_retreated', 'expr']
    )
    axes2[i].text(0.5, 0.95, f'p={pval:.4f}', transform=axes2[i].transAxes, ha='center', fontsize=9)
for j in range(len(dt_markers), len(axes2)):
    axes2[j].axis('off')
plt.suptitle('Figure 10. Markers, drug-holiday vs. re-treatment (depth-matched)')
plt.tight_layout()
plt.savefig('figure10_markers_reversibility.png', dpi=300, bbox_inches='tight')
plt.show()

---
## Code section 13 — §3.7: sensitivity of the CCM attribution to the QC threshold → Table 3, Figures 5 & 11

Everything above (sections 2–12) is repeated end to end as a single reusable
function, `run_downstream_pipeline`, so the same computation can be invoked
under a second, independently chosen QC threshold without copy-pasting the
pipeline. This directly reproduces the §3.7 sensitivity check: the trajectory
shape, correlation dimension (**Table 3, Figure 5**), and classification
accuracy are expected to be threshold-invariant, while the gene-level CCM
attribution (gene→trajectory direction only, for speed; **Figure 11**) is
expected to be threshold-sensitive for all but the two strongest drivers,
EGFR and SOX4.

**Note:** this cell reruns the full pipeline twice end to end (including 200
CCM surrogates per gene, per threshold) and will take a while.

In [ ]:
# §3.7 -- Reusable pipeline function, parameterised by the QC threshold.
def run_downstream_pipeline(adata_raw_in, min_counts, min_genes, max_mito,
                             dt_markers_wanted, condition_order, main_arm, day_map, seed=0):
    adata = adata_raw_in.copy()

    # QC -- now via sc.pp.calculate_qc_metrics(), matching §2.2 exactly, so
    # that calling this function with min_counts=300, min_genes=30,
    # max_mito=0.30 reproduces §2.2's cell counts exactly rather than
    # risking drift between two hand-written implementations.
    adata.var['mt'] = adata.var_names.str.upper().str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    adata.obs['n_genes_detected'] = adata.obs['n_genes_by_counts']
    adata.obs['mito_fraction'] = adata.obs['pct_counts_mt'] / 100.0
    tc, ng, mf = adata.obs['total_counts'].values, adata.obs['n_genes_detected'].values, adata.obs['mito_fraction'].values
    qc_mask = (tc >= min_counts) & (ng >= min_genes) & (mf < max_mito)
    aq = adata[qc_mask].copy()

    # Normalise, HVG, depth-regress, PCA
    sc.pp.normalize_total(aq, target_sum=1e4)
    sc.pp.log1p(aq)
    sc.pp.highly_variable_genes(aq, n_top_genes=2000, flavor='seurat')
    dtm = [g for g in dt_markers_wanted if g in aq.var_names]
    for gene in dtm:
        aq.var.loc[gene, 'highly_variable'] = True
    ah = aq[:, aq.var['highly_variable']].copy()
    ah.obs['log_total_counts'] = np.log1p(aq.obs['total_counts'].values)
    ah.obs['mito_fraction'] = aq.obs['mito_fraction'].values
    sc.pp.regress_out(ah, ['log_total_counts', 'mito_fraction'])
    sc.pp.scale(ah, max_value=10)
    sc.tl.pca(ah, n_comps=50, svd_solver='arpack')
    depth_r = np.corrcoef(ah.obsm['X_pca'][:, 0], ah.obs['log_total_counts'].values)[0, 1]

    # Trajectory
    d0m = (ah.obs['condition'] == 'D0').values
    d0c = ah.obsm['X_pca'][d0m].mean(axis=0)
    ah.obs['dist_from_baseline'] = np.linalg.norm(ah.obsm['X_pca'] - d0c, axis=1)
    dist_summary = ah.obs.groupby('condition', observed=True)['dist_from_baseline'].mean().reindex(main_arm)

    # Correlation dimension, main arm
    cdim = {}
    for cond in main_arm:
        mask = (ah.obs['condition'] == cond).values
        boot = bootstrap_corr_dim(ah.obsm['X_pca'][:, :15][mask], seed=seed)
        cdim[cond] = (mask.sum(), boot.mean(), np.percentile(boot, 2.5), np.percentile(boot, 97.5))

    # Depth-matching + reversibility correlation dimension
    ia, ib = depth_match_two_groups(aq, 'condition', 'D19_DMSO_holiday', 'D19_ERL_retreated', seed=seed)
    matched = np.concatenate([aq.obs_names[ia], aq.obs_names[ib]])
    arev = ah[ah.obs_names.isin(matched)].copy()
    for cond in ['D19_DMSO_holiday', 'D19_ERL_retreated']:
        mask = (arev.obs['condition'] == cond).values
        boot = bootstrap_corr_dim(arev.obsm['X_pca'][:, :15][mask], seed=seed)
        cdim[cond] = (mask.sum(), boot.mean(), np.percentile(boot, 2.5), np.percentile(boot, 97.5))

    # Classification
    Xc = arev.obsm['X_pca'][:, :20]
    yc = (arev.obs['condition'] == 'D19_ERL_retreated').astype(int).values
    cvk = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    clf = LogisticRegression(max_iter=1000)
    cvs = cross_val_score(clf, Xc, yc, cv=cvk)
    _, _, pval = permutation_test_score(clf, Xc, yc, cv=cvk, n_permutations=200, random_state=seed)

    # Marker-free axis
    basis_genes = [g for g in aq.var_names if g not in dtm]
    ab = aq[ah.obs_names, basis_genes].copy()
    sc.pp.highly_variable_genes(ab, n_top_genes=2000, flavor='seurat')
    abh = ab[:, ab.var['highly_variable']].copy()
    abh.obs['log_total_counts'] = np.log1p(aq.obs.loc[abh.obs_names, 'total_counts'].values)
    abh.obs['mito_fraction'] = aq.obs.loc[abh.obs_names, 'mito_fraction'].values
    sc.pp.regress_out(abh, ['log_total_counts', 'mito_fraction'])
    sc.pp.scale(abh, max_value=10)
    sc.tl.pca(abh, n_comps=50, svd_solver='arpack')
    d0c2 = abh.obs_names.isin(ah.obs_names[(ah.obs['condition'] == 'D0').values])
    centroid2 = abh.obsm['X_pca'][d0c2].mean(axis=0)
    dist_clean = pd.Series(np.linalg.norm(abh.obsm['X_pca'] - centroid2, axis=1), index=abh.obs_names)
    axis_corr = np.corrcoef(ah.obs['dist_from_baseline'].values, dist_clean.loc[ah.obs_names].values)[0, 1]

    # CCM (gene -> trajectory direction only, for speed)
    mam = ah.obs['condition'].isin(main_arm).values
    man = ah.obs_names[mam]
    rd = ah.obs.loc[man, 'condition'].map(day_map).values
    wdd = dist_clean.loc[man].values
    ok = rd + (wdd - wdd.min()) / (wdd.max() - wdd.min() + 1e-9)
    oi = np.argsort(ok)
    on = man[oi]
    nb = 30
    be = np.linspace(ok.min(), ok.max(), nb + 1)
    bi = np.clip(np.digitize(ok[oi], be) - 1, 0, nb - 1)
    axb = np.array([dist_clean.loc[on].values[bi == b].mean() if (bi == b).sum() > 0 else np.nan for b in range(nb)])

    ccm_out = {}
    for gene in dtm:
        gidx = np.where(aq.var_names == gene)[0][0]
        expr = pd.Series(np.asarray(aq[:, gidx].X.todense()).flatten(), index=aq.obs_names)
        eo = expr.loc[on].values
        gb = np.array([eo[bi == b].mean() if (bi == b).sum() > 0 else np.nan for b in range(nb)])
        binned = pd.DataFrame({'gene': gb, 'axis': axb}).interpolate(limit_direction='both')
        rxy, ryx, pxy, pyx = surrogate_test(binned['gene'].values, binned['axis'].values, seed=seed)
        ccm_out[gene] = {'gene_to_axis_rho': rxy, 'gene_to_axis_p': pxy,
                          'axis_to_gene_rho': ryx, 'axis_to_gene_p': pyx}

    return {
        'n_total_qc': int(qc_mask.sum()),
        'condition_counts': aq.obs['condition'].value_counts().reindex(condition_order),
        'dist_from_baseline': dist_summary,
        'depth_confound_corr': depth_r,
        'corr_dim': cdim,
        'classification_accuracy': cvs.mean(),
        'classification_std': cvs.std(),
        'permutation_p': pval,
        'marker_free_axis_corr': axis_corr,
        'ccm_results': ccm_out,
    }

print("run_downstream_pipeline() defined.")

In [ ]:
# §3.7 -- Run the pipeline under the original QC threshold (§2.2) and an
# independently chosen alternative threshold, spanning a comparable cell-count
# range, then compare.
results_original = run_downstream_pipeline(
    adata_raw, min_counts=300, min_genes=30, max_mito=0.30,
    dt_markers_wanted=DT_MARKERS_WANTED, condition_order=CONDITION_ORDER,
    main_arm=MAIN_ARM, day_map=DAY_MAP, seed=0,
)

results_alternative = run_downstream_pipeline(
    adata_raw, min_counts=250, min_genes=25, max_mito=0.35,
    dt_markers_wanted=DT_MARKERS_WANTED, condition_order=CONDITION_ORDER,
    main_arm=MAIN_ARM, day_map=DAY_MAP, seed=0,
)

print("Original threshold  -- total QC cells:", results_original['n_total_qc'])
print("Alternative threshold -- total QC cells:", results_alternative['n_total_qc'])
print()
print("Classification accuracy -- original: %.3f, alternative: %.3f" % (
    results_original['classification_accuracy'], results_alternative['classification_accuracy']))
print("Depth-confound correlation (after correction) -- original: %.3f, alternative: %.3f" % (
    results_original['depth_confound_corr'], results_alternative['depth_confound_corr']))
print("Marker-free axis correlation -- original: %.3f, alternative: %.3f" % (
    results_original['marker_free_axis_corr'], results_alternative['marker_free_axis_corr']))

In [ ]:
# Table 3 & Figure 5 -- correlation dimension by condition under the original
# and the alternative QC threshold, plotted together for direct comparison.
# Reuses results_original['corr_dim'] and results_alternative['corr_dim'],
# both of which already contain all eight conditions (main arm + the
# depth-matched reversibility pair) -- see run_downstream_pipeline() above.
table3 = pd.DataFrame(
    {c: results_alternative['corr_dim'][c] for c in CONDITION_ORDER},
    index=['n_cells', 'D2_mean', 'ci_lo', 'ci_hi']
).T
print("Table 3. Correlation dimension by condition, alternative QC threshold:")
print(table3)

means_orig = table1['D2_mean'].values
err_lo_orig = means_orig - table1['ci_lo'].values
err_hi_orig = table1['ci_hi'].values - means_orig
means_alt = table3['D2_mean'].values
err_lo_alt = means_alt - table3['ci_lo'].values
err_hi_alt = table3['ci_hi'].values - means_alt

x_pos = np.arange(len(CONDITION_ORDER))
offset = 0.12
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.errorbar(x_pos - offset, means_orig, yerr=[err_lo_orig, err_hi_orig], fmt='o', capsize=4,
            color='tab:blue', label='Original QC threshold')
ax.errorbar(x_pos + offset, means_alt, yerr=[err_lo_alt, err_hi_alt], fmt='o', capsize=4,
            color='tab:orange', label='Alternative QC threshold')
ax.axvline(5.5, linestyle=':', color='black')
ax.set_xticks(x_pos)
ax.set_xticklabels(CONDITION_ORDER, rotation=30, ha='right')
ax.set_ylabel('Correlation dimension (D2)')
ax.set_title('Figure 5. Correlation dimension, original vs. alternative QC threshold')
ax.legend()
plt.tight_layout()
plt.savefig('figure5_corr_dim_qc_sensitivity.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 11 -- CCM gene->trajectory skill under both QC thresholds.
genes_plot = [g for g in DT_MARKERS_WANTED if g in results_original['ccm_results'] and g in results_alternative['ccm_results']]
rho_orig = [results_original['ccm_results'][g]['gene_to_axis_rho'] for g in genes_plot]
p_orig = [results_original['ccm_results'][g]['gene_to_axis_p'] for g in genes_plot]
rho_alt = [results_alternative['ccm_results'][g]['gene_to_axis_rho'] for g in genes_plot]
p_alt = [results_alternative['ccm_results'][g]['gene_to_axis_p'] for g in genes_plot]

x = np.arange(len(genes_plot))
w = 0.36
fig, ax = plt.subplots(figsize=(8.0, 4.2))
ax.bar(x - w/2, rho_orig, width=w, color='#1F3864', label='Original QC threshold (§2.2)')
ax.bar(x + w/2, rho_alt, width=w, color='#C55A11', label='Alternative QC threshold')
for i in range(len(genes_plot)):
    if p_orig[i] < 0.05:
        ax.text(x[i] - w/2, rho_orig[i] + (0.03 if rho_orig[i] >= 0 else -0.07), '*', ha='center', fontsize=13, color='#1F3864')
    if p_alt[i] < 0.05:
        ax.text(x[i] + w/2, rho_alt[i] + (0.03 if rho_alt[i] >= 0 else -0.07), '*', ha='center', fontsize=13, color='#C55A11')
ax.axhline(0, color='#333333', lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(genes_plot, rotation=30, ha='right', fontstyle='italic')
ax.set_ylabel('CCM skill, gene -> trajectory (rho)')
ax.set_title('Figure 11. CCM gene-to-trajectory skill, original vs. alternative QC threshold')
ax.legend(frameon=False, fontsize=9, loc='upper left')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.grid(axis='y', linewidth=0.5, color='#E0E0E0', zorder=0)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('figure11_ccm_sensitivity.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nGene-level threshold stability (gene->trajectory direction):")
for g in genes_plot:
    stable = "stable" if (p_orig[genes_plot.index(g)] < 0.05) == (p_alt[genes_plot.index(g)] < 0.05) else "NOT stable"
    print(f"  {g}: original p={p_orig[genes_plot.index(g)]:.3f}, alternative p={p_alt[genes_plot.index(g)]:.3f}  -> {stable}")

---
## Code section 14 — permutation entropy & shuffle-surrogate test → Table 6, Figure 12

Tests whether the thirty-bin marker-free trajectory axis, and each marker
gene's binned expression, carries genuine sequential (order-dependent)
structure, as opposed to being statistically indistinguishable from a
randomly ordered set of the same points. Uses ordinal-pattern permutation
entropy (Bandt & Pompe, 2002), normalised to [0, 1], with a 200-shuffle
surrogate null — the same surrogate-testing convention already used for CCM
above — following the deterministic-vs-stochastic decision logic of Toker
et al. (2020). This does **not** distinguish chaos from other ordered
dynamics; it only establishes the weaker, necessary precondition that the
trajectory is not equivalent to randomly-ordered noise. Reuses `binned_df`
and `dt_markers` from Code sections 10–11 above — run those first.

In [ ]:
# Code section 14 -- Permutation entropy of the trajectory axis and each
# marker gene, with a 200-shuffle surrogate significance test.
import math

def permutation_entropy(x, m=3, tau=1):
    x = np.asarray(x)
    n = len(x)
    perm_counts = {}
    for i in range(n - (m - 1) * tau):
        window = x[i:i + m * tau:tau]
        pattern = tuple(np.argsort(window))
        perm_counts[pattern] = perm_counts.get(pattern, 0) + 1
    counts = np.array(list(perm_counts.values()), dtype=float)
    probs = counts / counts.sum()
    h = -np.sum(probs * np.log(probs))
    h_max = math.log(math.factorial(m))
    return h / h_max  # normalised to [0, 1]

def pe_surrogate_test(x, m=3, tau=1, n_surrogates=200, seed=0):
    rng = np.random.default_rng(seed)
    real_pe = permutation_entropy(x, m=m, tau=tau)
    surrogate_pe = np.array([
        permutation_entropy(rng.permutation(x), m=m, tau=tau)
        for _ in range(n_surrogates)
    ])
    # Real PE lower than the shuffle null -> genuine sequential structure.
    p_value = (surrogate_pe <= real_pe).mean()
    return real_pe, surrogate_pe.mean(), surrogate_pe.std(), p_value

series_to_test = {'trajectory axis': binned_df['axis'].values}
series_to_test.update({g: binned_df[g].values for g in dt_markers})

pe_rows = []
for name, series in series_to_test.items():
    real_pe, surr_mean, surr_sd, p_val = pe_surrogate_test(series, m=3, tau=1)
    pe_rows.append({'series': name, 'PE': real_pe, 'surrogate_PE_mean': surr_mean,
                     'surrogate_PE_sd': surr_sd, 'p_value': p_val})
    print(f"{name}: PE={real_pe:.3f}, surrogate PE={surr_mean:.3f}+/-{surr_sd:.3f}, "
          f"p={p_val:.3f}   <-- Table 6")

table4 = pd.DataFrame(pe_rows).set_index('series')  # variable name kept as table4; manuscript label is Table 6
print("\nTable 6. Permutation entropy, real vs. 200 shuffle-surrogates:")
print(table4)

In [ ]:
# Figure 12 -- permutation entropy: observed value vs. shuffle-surrogate null.
fig, ax = plt.subplots(figsize=(8, 4))
names = list(series_to_test.keys())
real_vals = [row['PE'] for row in pe_rows]
surr_means = [row['surrogate_PE_mean'] for row in pe_rows]
surr_sds = [row['surrogate_PE_sd'] for row in pe_rows]

x_pos = np.arange(len(names))
ax.errorbar(x_pos, surr_means, yerr=1.96 * np.array(surr_sds), fmt='o', color='gray',
            label='shuffle-surrogate null (mean $\\pm$ 95% CI)', capsize=4)
ax.scatter(x_pos, real_vals, color='tab:red', zorder=5, label='observed PE', s=60)
ax.set_xticks(x_pos)
ax.set_xticklabels(names, rotation=45, ha='right')
ax.set_ylabel('Normalised permutation entropy')
ax.set_title('Figure 12. Permutation entropy vs. shuffle-surrogate null')
ax.legend()
plt.tight_layout()
plt.savefig('figure12_permutation_entropy.png', dpi=300, bbox_inches='tight')
plt.show()

## Code section 15 — transfer entropy → Table 7, Figure 13

A second, independent causal-attribution estimate to compare against
convergent cross mapping (Table 2 / Figure 6 above). Uses a coarse
three-level (tertile) discretisation appropriate to the thirty-point binned
trajectory — finer binning would leave the joint-probability table too
sparse at this sample size — and the same 200-shuffle surrogate convention
used throughout. Reuses `binned_df`, `dt_markers`, and `table2` from Code
sections 10–11 above — run those first.

In [ ]:
# Code section 15 -- Discretised transfer entropy (Schreiber, 2000) between
# each marker gene and the marker-free trajectory axis, both directions.
def discretise(x, n_bins=3):
    edges = np.quantile(x, np.linspace(0, 1, n_bins + 1))
    edges[0] -= 1e-9
    edges[-1] += 1e-9
    return np.digitize(x, edges) - 1

def transfer_entropy(source, target, n_bins=3, lag=1):
    s = discretise(source, n_bins)[:-lag]
    t_now = discretise(target, n_bins)[lag:]
    t_past = discretise(target, n_bins)[:-lag]
    df = pd.DataFrame({'s': s, 't_now': t_now, 't_past': t_past})
    te = 0.0
    n = len(df)
    for (tp, tn, sp), grp in df.groupby(['t_past', 't_now', 's']):
        p_joint = len(grp) / n
        denom_tp_s = len(df[(df.t_past == tp) & (df.s == sp)])
        denom_tp = len(df[df.t_past == tp])
        if denom_tp_s == 0 or denom_tp == 0:
            continue
        p_tn_given_tp_s = len(grp) / denom_tp_s
        p_tn_given_tp = len(df[(df.t_past == tp) & (df.t_now == tn)]) / denom_tp
        if p_tn_given_tp_s > 0 and p_tn_given_tp > 0:
            te += p_joint * np.log(p_tn_given_tp_s / p_tn_given_tp)
    return te

def te_surrogate_test(source, target, n_bins=3, lag=1, n_surrogates=200, seed=0):
    rng = np.random.default_rng(seed)
    real_te = transfer_entropy(source, target, n_bins=n_bins, lag=lag)
    surr_te = np.array([
        transfer_entropy(rng.permutation(source), target, n_bins=n_bins, lag=lag)
        for _ in range(n_surrogates)
    ])
    p_val = (surr_te >= real_te).mean()
    return real_te, p_val

te_rows = []
for gene in dt_markers:
    te_g2a, p_g2a = te_surrogate_test(binned_df[gene].values, binned_df['axis'].values)
    te_a2g, p_a2g = te_surrogate_test(binned_df['axis'].values, binned_df[gene].values)
    te_rows.append({'gene': gene, 'TE_gene_to_axis': te_g2a, 'p_gene_to_axis': p_g2a,
                     'TE_axis_to_gene': te_a2g, 'p_axis_to_gene': p_a2g})
    print(f"{gene}: TE(gene->axis)={te_g2a:.3f} (p={p_g2a:.3f}), "
          f"TE(axis->gene)={te_a2g:.3f} (p={p_a2g:.3f})   <-- Table 7")

table5 = pd.DataFrame(te_rows).set_index('gene')  # variable name kept as table5; manuscript label is Table 7
print("\nTable 7. Transfer entropy vs. marker-free trajectory axis:")
print(table5)

In [ ]:
# Figure 13 -- transfer entropy (gene->axis) alongside CCM skill (gene->axis)
# for each marker gene, to visualise agreement between the two independent
# causal-attribution methods.
fig, ax = plt.subplots(figsize=(10, 4.5))
x_pos = np.arange(len(dt_markers))
width = 0.35

ccm_g2a = [table2.loc[g, 'gene_to_axis_rho'] for g in dt_markers]
te_g2a = [table5.loc[g, 'TE_gene_to_axis'] for g in dt_markers]

ax.bar(x_pos - width / 2, ccm_g2a, width, label='CCM skill (gene->axis)', color='tab:blue')
ax2 = ax.twinx()
ax2.bar(x_pos + width / 2, te_g2a, width, label='Transfer entropy (gene->axis)', color='tab:orange')

ax.set_xticks(x_pos)
ax.set_xticklabels(dt_markers, rotation=45, ha='right')
ax.set_ylabel('CCM skill (rho)', color='tab:blue')
ax2.set_ylabel('Transfer entropy (bits)', color='tab:orange')
ax.axhline(0, color='gray', linewidth=0.5)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=9)
plt.title('Figure 13. Transfer entropy vs. published CCM skill (gene -> axis)')

plt.tight_layout()
plt.savefig('figure13_te_vs_ccm.png', dpi=300, bbox_inches='tight')
plt.show()

## Code section 16 — quantitative Lyapunov-exponent data-sufficiency check (Discussion)

Turns "six timepoints is not enough for a Lyapunov exponent" into an actual
number, using the Eckmann–Ruelle (1992) heuristic $N_{\min}\approx10^{D/2}$
applied to this study's own measured correlation-dimension range from
Table 1 / `corr_dim_results` (Code sections 6–7 above — run those first).
These numbers are already incorporated into the Discussion paragraph on the Lyapunov-exponent data-sufficiency bound.

In [ ]:
# Code section 16 -- Eckmann-Ruelle data-sufficiency check for a Lyapunov
# exponent estimate, using this study's own measured D2 range.
d2_values = [v[1] for v in corr_dim_results.values()]
d2_min, d2_max = min(d2_values), max(d2_values)

n_min_lower = 10 ** (d2_min / 2)
n_min_upper = 10 ** (d2_max / 2)

n_real_timepoints = len(MAIN_ARM)          # 6 real biological timepoints
n_binned_trajectory = binned_df.shape[0]    # 30 pseudo-time bins used for CCM/PE/TE

print(f"Measured correlation dimension range across conditions: {d2_min:.2f} - {d2_max:.2f}")
print(f"Eckmann-Ruelle minimum N (N_min ~ 10^(D/2)) for reliable dimension/Lyapunov "
      f"estimation: {n_min_lower:.0f} - {n_min_upper:.0f} points   <-- Discussion")
print(f"Real timepoints available (main trajectory): {n_real_timepoints}")
print(f"Pseudo-time bins available (CCM/entropy trajectory): {n_binned_trajectory}")
print(f"Shortfall using real timepoints: {n_min_lower / n_real_timepoints:.1f}x to "
      f"{n_min_upper / n_real_timepoints:.1f}x below N_min")
print(f"Shortfall using the 30-bin trajectory: {n_min_lower / n_binned_trajectory:.1f}x to "
      f"{n_min_upper / n_binned_trajectory:.1f}x below N_min")

## Code section 17 — figure inventory check (all 13 manuscript figures)

Run this last. It simply lists every `figureN_*.png` file this notebook is
supposed to have saved to the working directory, in manuscript order, and
flags anything missing. Use this as the one-line proof, for a supervisor or
a collaborator, that all 13 figures in the manuscript were in fact produced by
this notebook and not added afterwards by hand.

In [ ]:
# Code section 17 -- confirm every manuscript figure was actually produced.
expected_figures = {
    1: 'figure1_pca_depth_correction.png',
    2: 'figure2_umap_trajectory.png',
    3: 'figure3_trajectory_dual_axis.png',
    4: 'figure4_correlation_dimension.png',
    5: 'figure5_corr_dim_qc_sensitivity.png',
    6: 'figure6_reversibility_umap.png',
    7: 'figure7_ccm_skill_bars.png',
    8: 'figure8_ccm_skill_curves.png',
    9: 'figure9_markers_trajectory.png',
    10: 'figure10_markers_reversibility.png',
    11: 'figure11_ccm_sensitivity.png',
    12: 'figure12_permutation_entropy.png',
    13: 'figure13_te_vs_ccm.png',
}

print(f"{'Fig.':<5}{'File':<38}{'Status'}")
all_present = True
for num, fname in expected_figures.items():
    ok = os.path.exists(fname)
    all_present &= ok
    print(f"{num:<5}{fname:<38}{'OK' if ok else 'MISSING -- rerun the cell above that saves it'}")

print()
print("All 13 manuscript figures present." if all_present else
      "Some figures are missing above -- scroll up and rerun the flagged cell(s).")

## Expert-feedback addendum A (Sept 2026) — §1: raw-expression check of the day-4 correlation-dimension minimum

One point raised in expert feedback on this analysis asks whether the $D_2$ minimum at day four reflects genuine
transcriptional convergence, or could instead be a by-product of (i) day four's
small cell count ($n=185$ after the unified QC, the smallest condition in the study) inflating the
geometric estimate's variance, or (ii) synchronisation limited to a subset of
genes rather than the full transcriptome. This cell checks both possibilities
directly in **raw, non-PCA expression space** — the corrected, scaled feature
matrix that PCA is built from (`adata_hvg2.X`, 2,009 genes) and, as an
assumption-lighter cross-check, the uncorrected log-normalised HVG expression
(`adata_qc2`) — using mean pairwise distance among cells, matched to the
same cell number (176) in every condition so the day-four sample-size
confound cannot drive the result either way.

This check has been run to completion; results are incorporated into the Discussion (see the paragraph on the day-four correlation-dimension minimum).

In [ ]:
# Expert-feedback addendum A -- raw-expression (non-PCA) dispersion check
# at day four, matched for cell number across all six main-trajectory
# conditions, to test the alternative interpretations raised in expert feedback.

def mean_pairwise_distance(mat, n_boot=200, sample_size=185, seed=0, metric='euclidean'):
    from scipy.spatial.distance import pdist
    rng = np.random.default_rng(seed)
    n = mat.shape[0]
    out = []
    for _ in range(n_boot):
        idx = rng.choice(n, size=min(sample_size, n), replace=(n < sample_size))
        out.append(pdist(mat[idx], metric=metric).mean())
    out = np.array(out)
    return out.mean(), np.percentile(out, 2.5), np.percentile(out, 97.5)

raw_expr_rows = []
for cond in MAIN_ARM:
    mask_hvg = (adata_hvg2.obs['condition'] == cond).values
    mask_qc = (adata_qc2.obs['condition'] == cond).values

    # (i) corrected, scaled expression -- the feature matrix PCA is built from
    corrected_mat = np.asarray(adata_hvg2.X[mask_hvg])
    mean_d_corr, lo_corr, hi_corr = mean_pairwise_distance(corrected_mat)

    # (ii) uncorrected log-normalised HVG expression -- assumption-light cross-check
    hvg_genes = adata_hvg2.var_names
    qc_hvg_idx = [np.where(adata_qc2.var_names == g)[0][0] for g in hvg_genes]
    raw_mat = np.asarray(adata_qc2.X[mask_qc][:, qc_hvg_idx].todense())
    mean_d_raw, lo_raw, hi_raw = mean_pairwise_distance(raw_mat)

    raw_expr_rows.append({
        'condition': cond, 'n_cells': mask_hvg.sum(),
        'mean_pairwise_dist_corrected': mean_d_corr,
        'ci_corrected': (lo_corr, hi_corr),
        'mean_pairwise_dist_raw_lognorm': mean_d_raw,
        'ci_raw_lognorm': (lo_raw, hi_raw),
    })
    print(f"{cond}: n={mask_hvg.sum():4d}  "
          f"corrected mean pairwise dist={mean_d_corr:.3f} [{lo_corr:.3f},{hi_corr:.3f}]  "
          f"raw log-norm mean pairwise dist={mean_d_raw:.3f} [{lo_raw:.3f},{hi_raw:.3f}]  "
          f"(both matched to n={min(176, mask_hvg.sum())} cells/condition, 200 bootstraps)")

table_R1 = pd.DataFrame(raw_expr_rows).set_index('condition')
print("\nSupplementary Table R1. Cell-number-matched raw-expression dispersion by condition:")
print(table_R1)
print(
    "\nInterpretation guide: if day four shows the lowest mean pairwise distance "
    "in BOTH the corrected and the raw log-normalised expression space even after "
    "matching every condition to the same 176 cells, that corroborates the "
    "in-transit-convergence reading in the manuscript. If the day-four dip "
    "disappears once cell number is matched, or appears only in the "
    "PCA-derived correlation-dimension estimate and not in raw expression "
    "distance, that instead supports the sample-size or PCA-artefact alternative "
    "raised in expert feedback, and the relevant Discussion paragraph should be revised "
    "accordingly."
)

## Expert-feedback addendum B (Sept 2026) — §2: a more stringent test of sequential order

Another point raised in expert feedback on this analysis asks whether permutation entropy at $m=3$ (only six
possible ordinal patterns) is a strong enough test of genuine deterministic
sequential order. Two independent strengthenings are added here, both applied
to the same marker-free trajectory axis and nine marker genes used throughout
(`binned_df`):

1. **Phase-randomised (autocorrelation-preserving) surrogates**, in place of
   pure label-shuffle surrogates, for the existing $m=3$ permutation entropy.
   A pure shuffle destroys all autocorrelation, so almost any smooth or
   correlated series looks "significant" against it. A Fourier phase-randomised
   surrogate preserves the series' power spectrum (hence its linear
   autocorrelation) while randomising phase, so a significant result against
   *this* null is a materially stronger claim: sequential structure beyond
   what a linear, equally autocorrelated process would produce by chance.
2. **A classical Wald–Wolfowitz runs test** on the sign of first differences —
   a completely different, non-ordinal-pattern statistical test for
   non-random sequential structure, used here to triangulate against the
   permutation-entropy result rather than restate it.

We also report permutation entropy at $m=4$ (24 possible patterns) against
the original pure-shuffle null, since the 30-point series can support it
(30 − 3 = 27 windows, comfortably above the $4! = 24$ pattern count),
giving a finer-grained version of the original test.

This check has been run to completion; results are incorporated into Limitations of Study (the phase-randomised and runs-test robustness checks).

In [ ]:
# Expert-feedback addendum B -- phase-randomised surrogate test, m=4
# permutation entropy, and a Wald-Wolfowitz runs test, as a more stringent
# check of genuine sequential order than the original m=3 pure-shuffle test.

from scipy.stats import norm

def phase_randomize(x, rng):
    '''Fourier phase-randomised surrogate: preserves the power spectrum
    (and hence linear autocorrelation) of x, randomises phase only.'''
    x = np.asarray(x, dtype=float)
    n = len(x)
    fx = np.fft.rfft(x - x.mean())
    amp = np.abs(fx)
    phases = rng.uniform(0, 2 * np.pi, size=len(fx))
    phases[0] = 0.0
    if n % 2 == 0:
        phases[-1] = 0.0
    fx_new = amp * np.exp(1j * phases)
    return np.fft.irfft(fx_new, n=n) + x.mean()

def pe_phase_surrogate_test(x, m=3, tau=1, n_surrogates=200, seed=0):
    rng = np.random.default_rng(seed)
    real_pe = permutation_entropy(x, m=m, tau=tau)
    surrogate_pe = np.array([
        permutation_entropy(phase_randomize(x, rng), m=m, tau=tau)
        for _ in range(n_surrogates)
    ])
    p_value = (surrogate_pe <= real_pe).mean()
    return real_pe, surrogate_pe.mean(), surrogate_pe.std(), p_value

def runs_test(x):
    '''Wald-Wolfowitz runs test on the sign of first differences. A low
    number of runs relative to the random-order expectation (negative z,
    small p) indicates non-random, trending sequential structure.'''
    diffs = np.diff(x)
    signs = np.sign(diffs)
    signs = signs[signs != 0]
    n1, n2 = (signs > 0).sum(), (signs < 0).sum()
    n = n1 + n2
    runs = 1 + np.sum(signs[1:] != signs[:-1])
    mean_runs = 2 * n1 * n2 / n + 1
    var_runs = (2 * n1 * n2 * (2 * n1 * n2 - n)) / (n ** 2 * (n - 1))
    z = (runs - mean_runs) / np.sqrt(var_runs)
    p = 2 * (1 - norm.cdf(abs(z)))
    return runs, z, p

series_to_test_R2 = {'trajectory axis': binned_df['axis'].values}
series_to_test_R2.update({g: binned_df[g].values for g in dt_markers})

r2_rows = []
for name, series in series_to_test_R2.items():
    pe4, surr4_mean, surr4_sd, p4 = pe_surrogate_test(series, m=4, tau=1)
    pe3_ph, surr_ph_mean, surr_ph_sd, p_ph = pe_phase_surrogate_test(series, m=3, tau=1)
    runs, z_runs, p_runs = runs_test(series)
    r2_rows.append({
        'series': name,
        'PE_m4': pe4, 'PE_m4_surrogate_mean': surr4_mean, 'PE_m4_p': p4,
        'PE_m3_phase_surrogate_mean': surr_ph_mean, 'PE_m3_phase_p': p_ph,
        'runs': runs, 'runs_z': z_runs, 'runs_p': p_runs,
    })
    print(f"{name}: "
          f"PE(m=4)={pe4:.3f} vs shuffle-null {surr4_mean:.3f}+/-{surr4_sd:.3f} (p={p4:.3f})  |  "
          f"PE(m=3) vs PHASE-RANDOMISED null {surr_ph_mean:.3f}+/-{surr_ph_sd:.3f} (p={p_ph:.3f})  |  "
          f"runs test z={z_runs:.2f} (p={p_runs:.3f})")

table_R2 = pd.DataFrame(r2_rows).set_index('series')
print("\nSupplementary Table R2. Stronger sequential-order tests (m=4 permutation entropy, "
      "phase-randomised-surrogate m=3 permutation entropy, and a Wald-Wolfowitz runs test):")
print(table_R2)
print(
    "\nInterpretation guide: the phase-randomised-surrogate p-value (PE_m3_phase_p) is the "
    "critical number for expert-feedback point 2. If it stays significant, the original m=3 "
    "shuffle-surrogate claim is substantially strengthened, since a much harder, "
    "autocorrelation-preserving null has also been cleared. If it loses significance while "
    "the pure-shuffle test (Figure 12 / Table 6) remained significant, the trajectory's "
    "apparent order is better explained by simple smoothness/autocorrelation than by genuine "
    "higher-order sequential structure, and the relevant Results paragraph and the paper's "
    "claim of 'genuine sequential structure' should be revised accordingly."
)

---
# Expert-feedback update (this notebook revision)

This revision responds to four points of feedback from a clinical oncology
colleague (relayed via Prof. Da Silva, Sept 2026):

1. Erlotinib is a first-generation EGFR-TKI; osimertinib (third-generation) is
   now first-line and clinically more relevant.
2. First-generation resistance is dominated by a single mutation (T790M,
   ~50% of cases); osimertinib resistance is more heterogeneous and less
   mechanistically uniform, which is arguably a *better* fit for a
   state-based / attractor-landscape framework.
3. Osimertinib is now frequently combined with platinum-pemetrexed
   chemotherapy in the clinic (FLAURA2), with an overall-survival benefit.
   Could that benefit relate to the tolerant-state population this paper
   describes?
4. Clinical re-challenge with the same TKI after a drug holiday shows very
   limited activity — does this contradict the paper's reversibility framing?

**What this revision adds, and what it deliberately does not attempt:**

- **Comments 1–2** are addressed by an independent replication arm on
  **GSE150949** (Oren *et al.*, *Nature* 2021), a PC9 + **osimertinib**
  time-course scRNA-seq dataset with lineage tracing (Watermelon system),
  added as **Section D** below. It uses the *same* estimators
  (correlation dimension, CCM, permutation entropy) already validated on
  GSE149383, so the two arms are directly comparable. GSE150949 has **no
  drug-holiday / re-treatment arm**, so it cannot replicate the bistability
  claim itself — only the trajectory-shape and driver-vs-marker split. This
  is stated explicitly in Section D and must be stated explicitly in the
  manuscript.
- **Comment 4** is addressed by a new **holiday-vs-D0** comparison on the
  existing GSE149383 data (**Section C1**, already flagged as a "concrete
  next step" in the original manuscript's Limitations section), plus a
  formal **bimodality test** (Hartigan's dip test / GMM, **Section C2**) on
  the reversibility arm, since the manuscript's non-overlapping-CI and
  classifier evidence are suggestive of two discrete states but do not
  formally establish it.
- **Comment 3** (chemotherapy + osimertinib) could **not** be given a
  matching public scRNA-seq dataset. No combination-therapy scRNA-seq study
  on EGFR-mutant persisters was found. This must remain a Discussion-level
  hypothesis in the manuscript, explicitly labelled as untested, rather than
  a new analysis. (GSE193258, a bulk RNA-seq osimertinib **washout**
  time-course in four cell lines including PC9, is a possible future
  orthogonal check on the reversibility-horizon question in Comment 4, but
  it is bulk, not single-cell, cannot carry the geometric/attractor analysis,
  and is not implemented here.)

**A reproducibility issue found in this notebook, fixed here:**
Cell 8's comment records that the quality-control step was already updated
once before, per an earlier round of expert feedback, to call `sc.pp.calculate_qc_metrics()`
instead of a hand-written QC computation. That fix was applied to the *code*
but the manuscript's Tables 1–3 and the "6,430 cells, unchanged under the
alternative threshold" claim were **not regenerated** afterwards, and no
longer match what this notebook produces (current original-threshold run:
D0 = 1,413 not 1,376; D4 = 185 not 176; alternative-threshold run: 6,961
cells total, not 6,430). Section **C0** below documents this plainly.
**Every number in the manuscript's Results, Tables 1–7 and Figures 1–13 must
be regenerated from a fresh, single, uninterrupted run of this notebook
before the manuscript is finalised** — do not hand-patch individual numbers.


## Section C0 — reproducibility note (read before regenerating tables)

Re-run cells 1–50 (the original GSE149383 pipeline) top to bottom in a single
fresh runtime. Do not reuse cached variables from an older run. The
authoritative cell counts, once QC is unified on `sc.pp.calculate_qc_metrics()`
as in the current cell 8, are printed by that cell. Table 1 (correlation
dimension) and Table 2 (CCM) will follow from those counts automatically.
Compare the printed totals against Table 1 and the "6,430 cells" claim in the
Summary/Results before submitting; if they still disagree, the discrepancy is
in the manuscript text, not in this code.

In [ ]:
# Section C0 (continued) -- Benjamini-Hochberg correction for the CCM and
# permutation-entropy families. table2 (CCM, 9 genes x 2 directions -> 18
# tests) and table4 (permutation entropy, axis + 9 genes -> 10 tests) already
# exist from Section 11 / Code section 14 above. This was described in the
# manuscript's Quantification and Statistical Analysis section but the
# actual correction was never implemented in the notebook -- added here.
from statsmodels.stats.multitest import multipletests

# --- CCM family (Table 2 / manuscript Table 4) ---
ccm_pvals = pd.concat([table2['gene_to_axis_p'].rename('p').to_frame().assign(direction='gene_to_axis'),
                        table2['axis_to_gene_p'].rename('p').to_frame().assign(direction='axis_to_gene')])
ccm_pvals = ccm_pvals.reset_index().rename(columns={'index': 'gene', 'gene': 'gene'})
reject, qvals, _, _ = multipletests(ccm_pvals['p'].values, alpha=0.05, method='fdr_bh')
ccm_pvals['q'] = qvals
ccm_pvals['significant_q05'] = reject

table2_q = table2.copy()
for gene in table2_q.index:
    table2_q.loc[gene, 'gene_to_axis_q'] = ccm_pvals.loc[
        (ccm_pvals['gene'] == gene) & (ccm_pvals['direction'] == 'gene_to_axis'), 'q'].values[0]
    table2_q.loc[gene, 'axis_to_gene_q'] = ccm_pvals.loc[
        (ccm_pvals['gene'] == gene) & (ccm_pvals['direction'] == 'axis_to_gene'), 'q'].values[0]

print("Table 2 with BH-corrected q-values (18 tests, q<0.05):")
print(table2_q[['gene_to_axis_rho', 'gene_to_axis_p', 'gene_to_axis_q',
                 'axis_to_gene_rho', 'axis_to_gene_p', 'axis_to_gene_q']])

# --- Permutation entropy family (Table 6) ---
reject_pe, qvals_pe, _, _ = multipletests(table4['p_value'].values, alpha=0.05, method='fdr_bh')
table4_q = table4.copy()
table4_q['q_value'] = qvals_pe
table4_q['significant_q05'] = reject_pe
print("\nTable 6 with BH-corrected q-values (10 tests, q<0.05):")
print(table4_q[['PE', 'p_value', 'q_value', 'significant_q05']])

# --- Transfer entropy family (Table 7), for completeness ---
te_pvals = pd.concat([table5['p_gene_to_axis'].rename('p'),
                       table5['p_axis_to_gene'].rename('p')])
reject_te, qvals_te, _, _ = multipletests(te_pvals.values, alpha=0.05, method='fdr_bh')
print(f"\nTransfer entropy family (18 tests): max q = {qvals_te.max():.3f} "
      f"(manuscript already reports all q > 0.46 by hand-computation; "
      f"this confirms it programmatically).")


### C0b — a stricter CCM convergence test (phase-randomised surrogates)

The manuscript's Discussion (Limitations) already notes that a pure
label-shuffle null is a weak bar for a smooth, thirty-point series, and
switches to a phase-randomised surrogate for the *permutation-entropy*
robustness check (Expert-feedback addendum B, cell 54). The same concern
applies to the CCM surrogate test in `surrogate_test()` above, which also
uses a plain shuffle. This cell repeats the CCM significance test with a
phase-randomised null (preserves each series' autocorrelation/power
spectrum, destroys only phase) as a harder, complementary bar. Genes that
survive *both* the shuffle and the phase-randomised test are the more
defensible candidate drivers.

In [ ]:
# Section C0b -- CCM surrogate test with a phase-randomised null, as a
# stricter complement to the plain-shuffle surrogate_test() used for Table 2.
def phase_randomize(x, seed=0):
    rng = np.random.default_rng(seed)
    n = len(x)
    fx = np.fft.rfft(x - x.mean())
    mag = np.abs(fx)
    phases = rng.uniform(0, 2 * np.pi, size=len(fx))
    phases[0] = 0.0
    if n % 2 == 0:
        phases[-1] = 0.0
    fx_new = mag * np.exp(1j * phases)
    return np.fft.irfft(fx_new, n=n) + x.mean()

def surrogate_test_phase(x, y, n_surrogates=200, seed=0):
    rng = np.random.default_rng(seed)
    sc_xy, sc_yx = run_ccm_pair(x, y)
    real_xy, real_yx = np.nanmax(sc_xy), np.nanmax(sc_yx)
    surr_xy, surr_yx = [], []
    for i in range(n_surrogates):
        try:
            y_s = phase_randomize(y, seed=seed * 10_000 + i)
            sxy, syx = run_ccm_pair(x, y_s)
            surr_xy.append(np.nanmax(sxy))
            surr_yx.append(np.nanmax(syx))
        except Exception:
            continue
    return real_xy, real_yx, (np.array(surr_xy) >= real_xy).mean(), (np.array(surr_yx) >= real_yx).mean()

table2_pr_rows = []
for gene in dt_markers:
    rxy, ryx, pxy, pyx = surrogate_test_phase(binned_df[gene].values, binned_df['axis'].values)
    table2_pr_rows.append({'gene': gene, 'gene_to_axis_rho': rxy, 'gene_to_axis_p_phaserand': pxy,
                            'axis_to_gene_rho': ryx, 'axis_to_gene_p_phaserand': pyx})
table2_phaserand = pd.DataFrame(table2_pr_rows).set_index('gene')

comparison = table2[['gene_to_axis_p', 'axis_to_gene_p']].join(
    table2_phaserand[['gene_to_axis_p_phaserand', 'axis_to_gene_p_phaserand']])
comparison['gene_to_axis_robust'] = (comparison['gene_to_axis_p'] < 0.05) & (comparison['gene_to_axis_p_phaserand'] < 0.05)
comparison['axis_to_gene_robust'] = (comparison['axis_to_gene_p'] < 0.05) & (comparison['axis_to_gene_p_phaserand'] < 0.05)
print("CCM significance under shuffle vs. phase-randomised surrogates:")
print(comparison)
print("\nGenes robust to BOTH null models, gene->axis direction:",
      comparison.index[comparison['gene_to_axis_robust']].tolist())


## Section C1 — drug-holiday vs. untreated (D0) baseline

Addresses expert-feedback point 4 (clinical re-challenge activity is limited)
and the manuscript's own Discussion / Limitations paragraph, which already
names this comparison as the natural next check: *"if holiday cells simply
revert most of the way to baseline rather than settling at a third, distinct
configuration, that would weaken the two-attractor reading further."*

Two possible outcomes and how to read them:
- **Holiday ≈ D0** (not significantly separable, D2 similar): supports true
  reversibility of the early persister state; the clinical failure of
  re-challenge is then attributable to *later*, genetic mechanisms (T790M,
  C797S, MET amplification, etc.) that arise after the window this dataset
  covers, not to a contradiction of the attractor model.
- **Holiday ≠ D0** (separable, distinct D2): supports a hysteresis /
  memory effect -- cells that have been through the drug do not fully
  reset even after withdrawal. This would *positively explain* the limited
  clinical benefit of re-challenge (the population never truly returns to
  the naive, fully drug-sensitive configuration) and strengthens, rather
  than weakens, the attractor-landscape reading. Either outcome is
  publishable; write it up as found, not as hoped for.

In [ ]:
# Section C1 -- classifier + correlation-dimension comparison, D19 DMSO
# holiday vs. D0, depth-matched exactly as done for the D19 holiday vs.
# D19 re-treated comparison in Section 7 / Section 9 above.
idx_d0, idx_holiday = depth_match_two_groups(adata_qc2, 'condition', 'D0', 'D19_DMSO_holiday')
matched_c1 = np.concatenate([adata_qc2.obs_names[idx_d0], adata_qc2.obs_names[idx_holiday]])
adata_c1 = adata_hvg2[adata_hvg2.obs_names.isin(matched_c1)].copy()
print(f"Depth-matched D0-vs-holiday set: {adata_c1.n_obs} cells "
      f"({(adata_c1.obs['condition']=='D0').sum()} D0, "
      f"{(adata_c1.obs['condition']=='D19_DMSO_holiday').sum()} holiday)")

# Correlation dimension, both groups, matched n
cdim_c1 = {}
for cond in ['D0', 'D19_DMSO_holiday']:
    mask = (adata_c1.obs['condition'] == cond).values
    boot = bootstrap_corr_dim(adata_c1.obsm['X_pca'][:, :15][mask], seed=0)
    cdim_c1[cond] = (mask.sum(), boot.mean(), np.percentile(boot, 2.5), np.percentile(boot, 97.5))
    print(f"{cond}: n={mask.sum()}, D2={boot.mean():.3f} [{np.percentile(boot,2.5):.3f}, {np.percentile(boot,97.5):.3f}]")

# Classifier
X_c1 = adata_c1.obsm['X_pca'][:, :20]
y_c1 = (adata_c1.obs['condition'] == 'D19_DMSO_holiday').astype(int).values
cv_c1 = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
clf_c1 = LogisticRegression(max_iter=1000)
cv_scores_c1 = cross_val_score(clf_c1, X_c1, y_c1, cv=cv_c1)
score_c1, _, pvalue_c1 = permutation_test_score(clf_c1, X_c1, y_c1, cv=cv_c1, n_permutations=200, random_state=0)
print(f"\nD0 vs. D19_DMSO_holiday classification: accuracy={cv_scores_c1.mean():.3f} "
      f"+/- {cv_scores_c1.std():.3f}, permutation p={pvalue_c1:.4f}")
print("Compare this accuracy against the 88.2% reported for holiday-vs-re-treated "
      "(Section 9 above). A much lower accuracy here (nearer 50%) supports the "
      "'holiday reverts toward baseline' reading; an accuracy comparable to 88.2% "
      "supports the hysteresis / third-state reading.")

table_c1 = pd.DataFrame(cdim_c1, index=['n_cells', 'D2_mean', 'ci_lo', 'ci_hi']).T
print("\nCorrelation dimension, D0 vs. holiday (new table for manuscript, e.g. Table 8):")
print(table_c1)


## Section C2 — formal bimodality test on the reversibility arm

The manuscript's Limitations section names Hartigan's dip test or a
two-component mixture model as a direct, not-yet-implemented next step to
formally test whether drug-holiday and re-treated cells form two discrete
modes rather than a shifted continuum. Implemented here on PC1 of the
depth-matched reversibility set (`adata_reversibility` from Section 7), plus
a 1-vs-2-component Gaussian-mixture BIC comparison on the top 2 PCs as a
second, independent bimodality check.

`diptest` needs to be installed: `pip install diptest` (already added to the
Colab setup cell at the top of this notebook -- see Setup section).

In [ ]:
# Section C2 -- Hartigan's dip test + GMM-BIC bimodality check on the
# depth-matched reversibility set (adata_reversibility, from Section 7 above).
import diptest
from sklearn.mixture import GaussianMixture

pc1_rev = adata_reversibility.obsm['X_pca'][:, 0]
dip_stat, dip_pval = diptest.diptest(pc1_rev)
print(f"Hartigan's dip test on PC1 (reversibility arm, n={len(pc1_rev)}): "
      f"D={dip_stat:.4f}, p={dip_pval:.4f}")
print("p < 0.05 rejects unimodality, i.e. supports two discrete modes.")

X_rev2 = adata_reversibility.obsm['X_pca'][:, :2]
bic1 = GaussianMixture(n_components=1, random_state=0).fit(X_rev2).bic(X_rev2)
bic2 = GaussianMixture(n_components=2, random_state=0).fit(X_rev2).bic(X_rev2)
print(f"\nGMM-BIC on top-2 PCs: 1-component BIC={bic1:.1f}, 2-component BIC={bic2:.1f}")
print(f"Lower BIC preferred -> {'2-component (bimodal)' if bic2 < bic1 else '1-component (unimodal)'} model favoured "
      f"(delta BIC = {bic1 - bic2:.1f}; a difference > 10 is usually read as decisive).")

gmm2 = GaussianMixture(n_components=2, random_state=0).fit(X_rev2)
cluster_labels = gmm2.predict(X_rev2)
true_labels = (adata_reversibility.obs['condition'] == 'D19_ERL_retreated').astype(int).values
agree = max((cluster_labels == true_labels).mean(), (cluster_labels != true_labels).mean())
print(f"Agreement between unsupervised 2-component GMM clusters and the real "
      f"holiday/re-treated labels: {agree:.3f} (compare against 0.882 classifier accuracy above; "
      f"close agreement means the two conditions ARE the two GMM modes, not that the GMM found "
      f"some other split).")


---
## Section D — independent replication in osimertinib-treated cells (GSE150949)

Continues directly from Section C above, reusing this notebook's existing
imports, marker gene list (`DT_MARKERS_WANTED`), and estimator functions
(`correlation_dimension`, `bootstrap_corr_dim`, `run_ccm_pair`,
`surrogate_test`, `permutation_entropy`, `pe_surrogate_test`) rather than
redefining them, so both arms use identical methods end to end.

GSE150949 (Oren et al., *Nature* 2021) profiles PC9 cells treated with
**osimertinib** (third-generation EGFR-TKI) across a fourteen-day time
course, with day-14 cells sorted by a lineage reporter into cycling,
moderately cycling, and non-cycling persisters. It has no drug-holiday /
re-treatment arm, so it cannot repeat the bistability test itself; it
provides an independent check of the trajectory shape, the gene-level causal
attribution, and a second, independent two-fate comparison (cycling vs.
non-cycling), addressed explicitly in Discussion.

**Memory note.** GSE150949 (56,419 cells) is substantially larger than any
single condition in the erlotinib arm. Three specific steps below are
written to stay memory-safe on modest hardware: the count-matrix parser uses
small chunks with explicit garbage collection, `scanpy.pp.regress_out` is
forced to run single-threaded (it multiprocesses by default, which
duplicates memory across worker processes), and the correlation-dimension
step subsamples to a fixed number of cells per day before computing
pairwise distances, since that computation scales quadratically and day 14
alone pools nearly 27,000 cells. Every step saves its results to
`/content/gse150949_results/` immediately, so a crash in a later step does
not lose earlier results.

In [ ]:
# Section D config + memory-reporting helpers. Everything else Section D
# needs (DT_MARKERS_WANTED, numpy/pandas/scanpy/sklearn/skccm imports,
# correlation_dimension, bootstrap_corr_dim, run_ccm_pair, surrogate_test,
# permutation_entropy, pe_surrogate_test) is already defined earlier in this
# notebook and is reused as-is, so both arms use identical methods.
import gc, resource, json as _json

os.makedirs('/content/gse150949', exist_ok=True)
RESULTS_DIR = '/content/gse150949_results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Memory-conscious settings for this arm (GSE150949 is ~7x larger than any
# single GSE149383 condition; see markdown above for why each of these
# matters). Adjust down further if you still hit an out-of-memory error.
N_TOP_GENES = 1000            # vs. 2,000 in the erlotinib arm; halved for memory. Flagged in Methods.
CSV_CHUNKSIZE = 300            # genes per chunk when parsing the ~123 MB count matrix
MAX_CELLS_FOR_CORRDIM = 1500   # per-day subsample cap for the O(n^2) correlation-dimension step only

def report_mem(label=""):
    mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024
    print(f"  [memory] peak RSS so far: {mb:,.0f} MB  {('-- ' + label) if label else ''}")

def save_result(name, obj):
    path_base = os.path.join(RESULTS_DIR, name)
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(path_base + ".csv")
        print(f"  [saved] {path_base}.csv")
    elif isinstance(obj, dict):
        with open(path_base + ".json", "w") as f:
            _json.dump(obj, f, indent=2, default=str)
        print(f"  [saved] {path_base}.json")
    else:
        raise ValueError(f"Don't know how to save type {type(obj)}")

report_mem("start of Section D")


## Step 1 — download + parse (small chunks, aggressive garbage collection)

Same two files as before (confirmed working): `GSE150949_metaData_with_lineage.txt.gz`
and `GSE150949_pc9_count_matrix.csv.gz`. The only change is `CSV_CHUNKSIZE=300`
(was 3000) and an explicit `del`+`gc.collect()` after every chunk, so peak
memory during parsing stays low instead of spiking to ~1.3 GB per chunk.
This step will take longer than before (more, smaller chunks) but should no
longer be the crash point.

In [ ]:
# Section D, step 1 -- download + low-memory chunked parse.
import urllib.request

BASE = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE150nnn/GSE150949/suppl/"

meta_path = "/content/gse150949/GSE150949_metaData_with_lineage.txt.gz"
if not os.path.exists(meta_path):
    urllib.request.urlretrieve(BASE + "GSE150949_metaData_with_lineage.txt.gz", meta_path)
meta_150949 = pd.read_csv(meta_path, sep='\t', index_col=0)
print("Metadata loaded:", meta_150949.shape)

mat_path = "/content/gse150949/GSE150949_pc9_count_matrix.csv.gz"
if not os.path.exists(mat_path):
    print("Downloading count matrix (~123 MB)...")
    urllib.request.urlretrieve(BASE + "GSE150949_pc9_count_matrix.csv.gz", mat_path)
    print("Download complete.")
else:
    print("Count matrix already downloaded, reusing local copy.")

print(f"Parsing in chunks of {CSV_CHUNKSIZE} genes (low-memory mode)...")
chunk_iter = pd.read_csv(mat_path, index_col=0, chunksize=CSV_CHUNKSIZE)
gene_chunks, sparse_chunks = [], []
cell_barcodes_150949 = None
for i, chunk in enumerate(chunk_iter):
    if cell_barcodes_150949 is None:
        cell_barcodes_150949 = chunk.columns.values
    gene_chunks.append(chunk.index.values.astype(str))
    sparse_chunks.append(sp.csr_matrix(chunk.values.astype(np.float32)))
    del chunk
    if i % 10 == 0:
        gc.collect()
        n_genes_so_far = sum(len(g) for g in gene_chunks)
        print(f"  ...chunk {i}, {n_genes_so_far} genes parsed so far")
        report_mem(f"chunk {i}")

gene_names_150949 = np.concatenate(gene_chunks)
del gene_chunks
gc.collect()

X_genes_by_cells = sp.vstack(sparse_chunks).tocsr()
del sparse_chunks
gc.collect()
print(f"Full matrix parsed: {X_genes_by_cells.shape[0]} genes x {X_genes_by_cells.shape[1]} cells")
report_mem("after full matrix assembled")

X_150949 = X_genes_by_cells.T.tocsr()
del X_genes_by_cells
gc.collect()

common_cells = pd.Index(cell_barcodes_150949).intersection(meta_150949.index)
print(f"{len(common_cells)} / {len(cell_barcodes_150949)} matrix cells matched in metadata.")
cell_pos = {bc: i for i, bc in enumerate(cell_barcodes_150949)}
keep_idx = np.array([cell_pos[bc] for bc in common_cells])
X_150949 = X_150949[keep_idx]
meta_aligned = meta_150949.loc[common_cells]

adata_150949_raw = ad.AnnData(X=X_150949)
adata_150949_raw.var_names = gene_names_150949
adata_150949_raw.var_names_make_unique()
adata_150949_raw.obs_names = common_cells

day_lookup = {0: 'D0', 3: 'D3', 7: 'D7', 14: 'D14'}
adata_150949_raw.obs['day'] = pd.Categorical(
    meta_aligned['time_point'].map(day_lookup).values,
    categories=['D0', 'D3', 'D7', 'D14'], ordered=True)

subpop_lookup = {'14_low': 'cycling', '14_med': 'moderate', '14_high': 'noncycling'}
adata_150949_raw.obs['subpopulation'] = meta_aligned['sample_type'].map(
    lambda s: subpop_lookup.get(s, 'bulk')).values

adata_150949_raw.obs['lineage_barcode'] = meta_aligned['lineage_barcode'].values
adata_150949_raw.obs['clone_size'] = meta_aligned['clone_size'].values
adata_150949_raw.obs['sample_type_raw'] = meta_aligned['sample_type'].values

del X_150949, meta_150949, meta_aligned
gc.collect()

print(adata_150949_raw)
crosstab_d1 = pd.crosstab(adata_150949_raw.obs['day'], adata_150949_raw.obs['subpopulation'])
print("\nday x subpopulation cross-tab:")
print(crosstab_d1)
save_result("step1_day_subpop_crosstab", crosstab_d1)
report_mem("end of step 1")


## Step 2 — QC, normalisation, depth correction (memory-safe)

Changes from before: `N_TOP_GENES=1000` (was 2000), `regress_out(..., n_jobs=1)`
to stop scanpy from multiprocessing (which duplicates memory across worker
processes -- likely the second-biggest crash risk after the `pdist` bug),
and `adata_150949_raw` is deleted immediately after QC since it's no longer
needed.

In [ ]:
# Section D, step 2 -- QC + normalise + HVG + depth-regress + PCA.
MIN_COUNTS_150949, MIN_GENES_150949, MAX_MITO_150949 = 300, 30, 0.30

adata_150949_raw.var['mt'] = adata_150949_raw.var_names.str.upper().str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata_150949_raw, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
adata_150949_raw.obs['mito_fraction'] = adata_150949_raw.obs['pct_counts_mt'] / 100.0

qc_mask_150949 = (
    (adata_150949_raw.obs['total_counts'] >= MIN_COUNTS_150949)
    & (adata_150949_raw.obs['n_genes_by_counts'] >= MIN_GENES_150949)
    & (adata_150949_raw.obs['mito_fraction'] < MAX_MITO_150949)
).values
adata_150949_qc = adata_150949_raw[qc_mask_150949].copy()

globals().pop('adata_150949_raw', None)  # safe: no error if already deleted
gc.collect()
report_mem("after QC, raw object freed")

print(f"{qc_mask_150949.sum()} / {len(qc_mask_150949)} cells passed QC")
qc_group_counts = adata_150949_qc.obs.groupby(['day', 'subpopulation'], observed=True).size()
print(qc_group_counts)
save_result("step2_qc_group_counts", qc_group_counts.reset_index(name='n_cells'))

sc.pp.normalize_total(adata_150949_qc, target_sum=1e4)
sc.pp.log1p(adata_150949_qc)
sc.pp.highly_variable_genes(adata_150949_qc, n_top_genes=N_TOP_GENES, flavor='seurat')

dt_markers_150949 = [g for g in DT_MARKERS_WANTED if g in adata_150949_qc.var_names]
missing_markers_150949 = sorted(set(DT_MARKERS_WANTED) - set(dt_markers_150949))
print(f"\nMarker genes available: {dt_markers_150949}")
if missing_markers_150949:
    print(f"Marker genes NOT found (excluded): {missing_markers_150949}")

for gene in dt_markers_150949:
    adata_150949_qc.var.loc[gene, 'highly_variable'] = True
adata_150949_hvg = adata_150949_qc[:, adata_150949_qc.var['highly_variable']].copy()
gc.collect()
report_mem(f"after HVG subset ({adata_150949_hvg.n_vars} genes)")

adata_150949_hvg.obs['log_total_counts'] = np.log1p(adata_150949_qc.obs['total_counts'].values)
adata_150949_hvg.obs['mito_fraction'] = adata_150949_qc.obs['mito_fraction'].values

_tmp_pca_check_150949 = adata_150949_hvg.copy()
sc.tl.pca(_tmp_pca_check_150949, n_comps=1, svd_solver='arpack')
depth_corr_before_150949 = np.corrcoef(
    _tmp_pca_check_150949.obsm['X_pca'][:, 0],
    adata_150949_hvg.obs['log_total_counts'].values)[0, 1]
del _tmp_pca_check_150949
gc.collect()

# n_jobs=1 -- scanpy's regress_out multiprocesses by default, which
# duplicates data across worker processes and can exhaust a 4GB machine.
sc.pp.regress_out(adata_150949_hvg, ['log_total_counts', 'mito_fraction'], n_jobs=1)
gc.collect()
report_mem("after regress_out")

sc.pp.scale(adata_150949_hvg, max_value=10)
sc.tl.pca(adata_150949_hvg, n_comps=50, svd_solver='arpack')
gc.collect()
report_mem("after scale + PCA")

depth_corr_after_150949 = np.corrcoef(
    adata_150949_hvg.obsm['X_pca'][:, 0], adata_150949_hvg.obs['log_total_counts'].values)[0, 1]
print(f"\nPC1-depth correlation before / after regression: "
      f"{depth_corr_before_150949:.3f} -> {depth_corr_after_150949:.3f}")

save_result("step2_depth_correlation", {
    "before": float(depth_corr_before_150949),
    "after": float(depth_corr_after_150949),
    "n_top_genes_used": N_TOP_GENES,
    "markers_available": dt_markers_150949,
    "markers_missing": missing_markers_150949,
})


## Step 3 — trajectory shape (all cells, cheap) + correlation dimension (capped, safe)

Distance-from-baseline uses **all** cells per day (O(n), no memory risk).
Correlation dimension is computed on a **fixed-size random subsample
(`MAX_CELLS_FOR_CORRDIM`) per day** — this is the fix for the `pdist` O(n²)
crash. Day 14 pools all three sorted subpopulations, same as before.

In [ ]:
# Section D, step 3 -- trajectory distance (all cells) + correlation
# dimension (capped subsample per day, to avoid the O(n^2) pdist blowup).
day14_mask_150949 = (adata_150949_hvg.obs['day'] == 'D14').values
bulk_mask_150949 = (
    (adata_150949_hvg.obs['subpopulation'] == 'bulk') | day14_mask_150949
).values
adata_150949_bulk = adata_150949_hvg[bulk_mask_150949].copy()
print(f"Trajectory arm: {adata_150949_bulk.n_obs} cells "
      f"({(adata_150949_bulk.obs['day']=='D0').sum()} D0, "
      f"{(adata_150949_bulk.obs['day']=='D3').sum()} D3, "
      f"{(adata_150949_bulk.obs['day']=='D7').sum()} D7, "
      f"{(adata_150949_bulk.obs['day']=='D14').sum()} D14 [3 sorts pooled])")

d0_mask_150949 = (adata_150949_bulk.obs['day'] == 'D0').values
d0_centroid_150949 = adata_150949_bulk.obsm['X_pca'][d0_mask_150949].mean(axis=0)
adata_150949_bulk.obs['dist_from_baseline'] = np.linalg.norm(
    adata_150949_bulk.obsm['X_pca'] - d0_centroid_150949, axis=1)

DAY_ORDER_150949 = ['D0', 'D3', 'D7', 'D14']
dist_by_day_150949 = adata_150949_bulk.obs.groupby('day', observed=True)['dist_from_baseline'].mean().reindex(DAY_ORDER_150949)
print("\nMean distance from D0 centroid, by day (all cells, no subsampling):")
print(dist_by_day_150949)
save_result("step3_distance_by_day", dist_by_day_150949.reset_index(name='mean_distance'))

# --- correlation dimension, CAPPED subsample per day (the actual fix) ---
rng_corrdim = np.random.default_rng(0)
table_d3_cdim = {}
for day in DAY_ORDER_150949:
    mask = (adata_150949_bulk.obs['day'] == day).values
    idx_full = np.where(mask)[0]
    if len(idx_full) > MAX_CELLS_FOR_CORRDIM:
        idx_use = rng_corrdim.choice(idx_full, size=MAX_CELLS_FOR_CORRDIM, replace=False)
        print(f"{day}: {len(idx_full)} cells available, subsampled to {MAX_CELLS_FOR_CORRDIM} for correlation dimension")
    else:
        idx_use = idx_full
    if len(idx_use) < 15:
        print(f"{day}: only {len(idx_use)} cells, skipping (needs >=15).")
        continue
    points = adata_150949_bulk.obsm['X_pca'][:, :15][idx_use]
    boot = bootstrap_corr_dim(points, seed=0)
    table_d3_cdim[day] = (len(idx_use), boot.mean(), np.percentile(boot, 2.5), np.percentile(boot, 97.5))
    print(f"{day}: n_used={len(idx_use)}, D2={boot.mean():.3f} [{np.percentile(boot,2.5):.3f}, {np.percentile(boot,97.5):.3f}]")
    gc.collect()

table_d3 = pd.DataFrame(table_d3_cdim, index=['n_cells_used', 'D2_mean', 'ci_lo', 'ci_hi']).T
print("\nTable D1 (new, for manuscript): correlation dimension by day, GSE150949:")
print(table_d3)
save_result("table_d3_correlation_dimension", table_d3)
report_mem("after correlation dimension")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(DAY_ORDER_150949, dist_by_day_150949.values, 'o-', color='tab:green')
ax.set_xlabel('Day (osimertinib)')
ax.set_ylabel('Mean distance from D0 centroid')
ax.set_title('Figure D1. Trajectory shape, GSE150949 (osimertinib)')
plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, 'figureD1_osimertinib_trajectory.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"  [saved] {fig_path}")


## Step 3b — diffusion pseudotime (second, independent trajectory measure)

The erlotinib arm's Figure 3 plots **two** independently computed trajectory
measures side by side: Euclidean distance from the D0 centroid (already
computed in step 3 above) and diffusion pseudotime, a manifold/random-walk-based
measure built from a k=15 nearest-neighbour graph. The point of showing both
together is that two measures built on very different geometric assumptions
landing on the same shape is much stronger evidence that the shape is real
biology, not an artefact of one method's assumptions. The osimertinib arm was
missing this second measure -- this step adds it, exactly mirroring the
erlotinib recipe (same k=15, same root-cell-nearest-to-D0-centroid convention).

This only needs a k-nearest-neighbour graph and a 15-component diffusion map,
not a full UMAP embedding, so it is cheap and should not reintroduce any
memory risk -- unlike the correlation-dimension step, this does not require
pairwise distances between all cells.

In [ ]:
# Section D, step 3b -- diffusion pseudotime, second independent trajectory
# measure, mirroring the erlotinib arm's Figure 3 exactly (Code section 5 in
# that notebook: sc.pp.neighbors -> sc.tl.diffmap -> sc.tl.dpt, root cell =
# the cell nearest the D0 centroid in PCA space). Reuses adata_150949_bulk,
# d0_mask_150949, d0_centroid_150949, DAY_ORDER_150949 from step 3 above --
# run this AFTER step 3, in the same session, not standalone.
sc.pp.neighbors(adata_150949_bulk, n_neighbors=15, use_rep='X_pca')
sc.tl.diffmap(adata_150949_bulk, n_comps=15)

root_idx_150949 = np.where(d0_mask_150949)[0][
    np.argmin(np.linalg.norm(adata_150949_bulk.obsm['X_pca'][d0_mask_150949] - d0_centroid_150949, axis=1))
]
adata_150949_bulk.uns['iroot'] = root_idx_150949
sc.tl.dpt(adata_150949_bulk)
gc.collect()
report_mem("after diffusion pseudotime")

dpt_by_day_150949 = adata_150949_bulk.obs.groupby('day', observed=True)['dpt_pseudotime'].mean().reindex(DAY_ORDER_150949)
dpt_sem_150949 = adata_150949_bulk.obs.groupby('day', observed=True)['dpt_pseudotime'].sem().reindex(DAY_ORDER_150949)
dist_sem_150949 = adata_150949_bulk.obs.groupby('day', observed=True)['dist_from_baseline'].sem().reindex(DAY_ORDER_150949)

print("Mean diffusion pseudotime, by day (osimertinib):")
print(dpt_by_day_150949)

table_step3b = pd.DataFrame({
    'dist_mean': dist_by_day_150949, 'dist_sem': dist_sem_150949,
    'dpt_mean': dpt_by_day_150949, 'dpt_sem': dpt_sem_150949,
})
save_result("step3b_dual_trajectory_measures", table_step3b)

# Rebuild Figure D1 as a dual-axis plot, mirroring the erlotinib arm's Figure 3.
x_pos = np.arange(len(DAY_ORDER_150949))
fig, ax1 = plt.subplots(figsize=(7, 5))
ax1.errorbar(x_pos, dist_by_day_150949.values, yerr=dist_sem_150949.values, marker='o', color='tab:blue',
             label='Distance from baseline (left axis)')
ax1.set_xlabel('Day (osimertinib)')
ax1.set_ylabel('Mean distance from D0 centroid', color='tab:blue')
ax1.set_xticks(x_pos); ax1.set_xticklabels(DAY_ORDER_150949)
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()
ax2.errorbar(x_pos, dpt_by_day_150949.values, yerr=dpt_sem_150949.values, marker='s', linestyle='--', color='tab:red',
             label='Diffusion pseudotime (right axis)')
ax2.set_ylabel('Mean diffusion pseudotime', color='tab:red')
ax2.tick_params(axis='y', labelcolor='tab:red')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=8)
plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, 'figureD1_osimertinib_trajectory_dual.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"  [saved] {fig_path}")


In [ ]:
# Section D, step 4 -- classifier: cycling vs. non-cycling day-14 persisters.
d14_sorted_mask = adata_150949_hvg.obs['subpopulation'].isin(['cycling', 'noncycling']).values
adata_d14 = adata_150949_hvg[d14_sorted_mask].copy()
print(adata_d14.obs['subpopulation'].value_counts())

classifier_result_d4 = {}
if adata_d14.n_obs >= 30 and adata_d14.obs['subpopulation'].nunique() == 2:
    X_d14 = adata_d14.obsm['X_pca'][:, :20]
    y_d14 = (adata_d14.obs['subpopulation'] == 'cycling').astype(int).values
    cv_d14 = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    clf_d14 = LogisticRegression(max_iter=1000)
    cv_scores_d14 = cross_val_score(clf_d14, X_d14, y_d14, cv=cv_d14)
    score_d14, _, pvalue_d14 = permutation_test_score(clf_d14, X_d14, y_d14, cv=cv_d14, n_permutations=200, random_state=0)
    print(f"\nCycling vs. non-cycling (day 14) classification: "
          f"accuracy={cv_scores_d14.mean():.3f} +/- {cv_scores_d14.std():.3f}, "
          f"permutation p={pvalue_d14:.4f}")
    classifier_result_d4 = {
        "accuracy_mean": float(cv_scores_d14.mean()),
        "accuracy_std": float(cv_scores_d14.std()),
        "permutation_p": float(pvalue_d14),
        "n_cycling": int((y_d14 == 1).sum()),
        "n_noncycling": int((y_d14 == 0).sum()),
    }
else:
    print("Not enough cells/classes in the sorted day-14 subsets to classify.")
    classifier_result_d4 = {"status": "skipped, insufficient cells/classes"}

save_result("step4_cycling_classifier", classifier_result_d4)
del adata_d14
gc.collect()
report_mem("after classifier")


## Step 5 — CCM + permutation entropy on the marker-free trajectory axis

**`adata_150949_hvg` is deleted before this step.** It's not needed anymore
(steps 3-4 already used it) and this step rebuilds its own, separate
~1000-gene basis matrix -- keeping both dense objects in memory at once
would roughly double peak RAM for no reason. This was likely a meaningful
contributor to earlier crashes.

In [ ]:
# Section D, step 5 -- marker-free trajectory axis, then CCM + permutation
# entropy against it. adata_150949_hvg freed first (see markdown above).
globals().pop('adata_150949_hvg', None)  # safe: no error if already deleted
gc.collect()
report_mem("after freeing adata_150949_hvg, before rebuilding basis")

genes_for_basis_150949 = [g for g in adata_150949_qc.var_names if g not in dt_markers_150949]
adata_basis_150949 = adata_150949_qc[adata_150949_bulk.obs_names, genes_for_basis_150949].copy()
sc.pp.highly_variable_genes(adata_basis_150949, n_top_genes=N_TOP_GENES, flavor='seurat')
adata_basis_150949_hvg = adata_basis_150949[:, adata_basis_150949.var['highly_variable']].copy()
del adata_basis_150949
gc.collect()

adata_basis_150949_hvg.obs['log_total_counts'] = np.log1p(
    adata_150949_qc.obs.loc[adata_basis_150949_hvg.obs_names, 'total_counts'].values)
adata_basis_150949_hvg.obs['mito_fraction'] = adata_150949_qc.obs.loc[
    adata_basis_150949_hvg.obs_names, 'mito_fraction'].values
sc.pp.regress_out(adata_basis_150949_hvg, ['log_total_counts', 'mito_fraction'], n_jobs=1)
gc.collect()
report_mem("after basis regress_out")

sc.pp.scale(adata_basis_150949_hvg, max_value=10)
sc.tl.pca(adata_basis_150949_hvg, n_comps=50, svd_solver='arpack')
gc.collect()
report_mem("after basis scale + PCA")

d0_mask_clean_150949 = adata_basis_150949_hvg.obs_names.isin(
    adata_150949_bulk.obs_names[(adata_150949_bulk.obs['day'] == 'D0').values])
centroid_clean_150949 = adata_basis_150949_hvg.obsm['X_pca'][d0_mask_clean_150949].mean(axis=0)
dist_clean_150949 = pd.Series(
    np.linalg.norm(adata_basis_150949_hvg.obsm['X_pca'] - centroid_clean_150949, axis=1),
    index=adata_basis_150949_hvg.obs_names)

day_map_150949 = {'D0': 0, 'D3': 3, 'D7': 7, 'D14': 14}
real_day_150949 = adata_150949_bulk.obs.loc[adata_basis_150949_hvg.obs_names, 'day'].astype(str).map(day_map_150949).astype(np.float64).values
within_day_dist_150949 = dist_clean_150949.values
order_key_150949 = real_day_150949 + (
    (within_day_dist_150949 - within_day_dist_150949.min())
    / (within_day_dist_150949.max() - within_day_dist_150949.min() + 1e-9))
order_idx_150949 = np.argsort(order_key_150949)
ordered_names_150949 = adata_basis_150949_hvg.obs_names[order_idx_150949]

n_bins_150949 = 30
bin_edges_150949 = np.linspace(order_key_150949.min(), order_key_150949.max(), n_bins_150949 + 1)
bin_idx_150949 = np.clip(np.digitize(order_key_150949[order_idx_150949], bin_edges_150949) - 1, 0, n_bins_150949 - 1)

axis_binned_150949 = np.array([
    dist_clean_150949.loc[ordered_names_150949].values[bin_idx_150949 == b].mean()
    if (bin_idx_150949 == b).sum() > 0 else np.nan
    for b in range(n_bins_150949)
])

gene_binned_150949 = {}
for gene in dt_markers_150949:
    gidx = np.where(adata_150949_qc.var_names == gene)[0][0]
    expr = pd.Series(np.asarray(adata_150949_qc[:, gidx].X.todense()).flatten(),
                      index=adata_150949_qc.obs_names)
    expr_ordered = expr.loc[ordered_names_150949].values
    gene_binned_150949[gene] = np.array([
        expr_ordered[bin_idx_150949 == b].mean() if (bin_idx_150949 == b).sum() > 0 else np.nan
        for b in range(n_bins_150949)
    ])

binned_df_150949 = pd.DataFrame({**gene_binned_150949, 'axis': axis_binned_150949}).interpolate(limit_direction='both')
save_result("binned_df_150949", binned_df_150949)

del adata_basis_150949_hvg
gc.collect()
report_mem("after building binned trajectory (basis object freed)")

table_d5_rows = []
for gene in dt_markers_150949:
    real_xy, real_yx, p_xy, p_yx = surrogate_test(binned_df_150949[gene].values, binned_df_150949['axis'].values)
    table_d5_rows.append({'gene': gene, 'gene_to_axis_rho': real_xy, 'gene_to_axis_p': p_xy,
                           'axis_to_gene_rho': real_yx, 'axis_to_gene_p': p_yx})
    print(f"{gene}: gene->axis={real_xy:.3f} (p={p_xy:.3f}) axis->gene={real_yx:.3f} (p={p_yx:.3f})")

table_d5 = pd.DataFrame(table_d5_rows).set_index('gene')
reject_d5, qvals_d5, _, _ = multipletests(
    pd.concat([table_d5['gene_to_axis_p'], table_d5['axis_to_gene_p']]).values, alpha=0.05, method='fdr_bh')
print(f"\nBH-corrected: max q = {qvals_d5.max():.3f}, {reject_d5.sum()} / {len(reject_d5)} tests significant at q<0.05")
print("\nTable D2 (new, for manuscript): CCM results, GSE150949 (osimertinib):")
print(table_d5)
save_result("table_d5", table_d5)

series_to_test_150949 = {'trajectory axis (osimertinib)': binned_df_150949['axis'].values}
series_to_test_150949.update({g: binned_df_150949[g].values for g in dt_markers_150949})
pe_rows_150949 = []
for name, series in series_to_test_150949.items():
    real_pe, surr_mean, surr_sd, p_val = pe_surrogate_test(series, m=3, tau=1)
    pe_rows_150949.append({'series': name, 'PE': real_pe, 'surrogate_PE_mean': surr_mean,
                            'surrogate_PE_sd': surr_sd, 'p_value': p_val})
table_d5_pe = pd.DataFrame(pe_rows_150949).set_index('series')
print("\nTable D3 (new, for manuscript): permutation entropy, GSE150949 (osimertinib):")
print(table_d5_pe)
save_result("table_d5_pe", table_d5_pe)
report_mem("end of Section D")


## Final — zip results and download

Zips everything in `/content/gse150949_results/` and triggers a browser
download (Colab only; if this isn't Colab, the folder path is printed so
you can grab it manually).

In [ ]:
# Zip and download all results.
import shutil
zip_path = shutil.make_archive('/content/gse150949_results', 'zip', RESULTS_DIR)
print(f"Zipped: {zip_path}")
print("\nContents:")
for f in sorted(os.listdir(RESULTS_DIR)):
    print(" ", f)

try:
    from google.colab import files
    files.download(zip_path)
    print("\nDownload triggered -- check your browser's downloads.")
except ImportError:
    print(f"\nNot running in Colab -- find your results at: {zip_path}")


---
# SECTION E — Analyses added in response to the pre-submission review

Everything below is **new** and is called from `reviewer_controls.py` (upload it next to this notebook, or
clone the repository). Run it **after** the cells above in the same session (it re-uses `adata_raw`,
`adata_qc2`, `adata_hvg2`, `adata_reversibility`, `binned_df`, `dt_markers`, `run_ccm_pair`, `MAIN_ARM`, `DAY_MAP`).

| Review point | Cell | Output to paste into the manuscript |
|---|---|---|
| 2 (batch confounding: D11 library vs library, D0 vs holiday / re-treated, covariate-only classifier, AUC) | E2 | Table 12 (`table_E_controls.csv`) |
| 2 / 5f (PC-count sensitivity of the classifier) | E3 | Supplementary table |
| 5 (D2 at equal n, common radii, i != j, log C(r) vs log r, direct D2 difference, PC sensitivity) | E4 | Table 3 (equal n), Figure S-D2, text |
| 8 (radial depth effect, count thinning, Scrublet) | E5 | Table 13 |
| 4 (how many bins are empty, no-interpolation rerun, twin / IAAFT surrogates, matched random-gene control, CCM settings) | E6 | Tables 4/9 caption, Methods, text |
| 6 (BH q-values with p = (r+1)/(M+1); Table 5 q-values) | E7 | Tables 4, 5 |
| 6 (alternative-QC rerun really used a different threshold / cell set / seed) | E8 | Methods sentence |

## E0 — create the helper module `reviewer_controls.py`  (this is what was missing)

In [ ]:
%%writefile /content/reviewer_controls.py
"""
reviewer_controls.py  --  helper functions for SECTION E of
"GSE149383 / GSE150949 drug-tolerance trajectory analysis".

Only needs packages that the main notebook already installs
(numpy, scipy, pandas, scikit-learn, matplotlib).  scikit-image and
`scrublet` are used if they happen to be importable, but are NOT required.

Every function below is called from Section E (cells E1-E8) of the notebook.
p-value convention used everywhere:  p = (r + 1) / (M + 1)
    r = number of null draws >= (or <=) the observed value, M = number of draws.
"""
import math
import warnings

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
__version__ = "1.0"

# ----------------------------------------------------------------------------
# small utilities
# ----------------------------------------------------------------------------
N_RADII_DEFAULT = 20
SCALING_FRAC_DEFAULT = (0.15, 0.60)   # same fitting window as the original estimator
LAST_DOUBLET_METHOD = {}              # filled by scrublet_by_library(): library -> method used


def _p_plus_one(n_ge, n_total):
    return (n_ge + 1.0) / (n_total + 1.0)


def benjamini_hochberg(pvals):
    """Benjamini-Hochberg FDR q-values, returned in the ORIGINAL order of `pvals`."""
    p = np.asarray(pvals, dtype=float)
    q = np.full(p.shape, np.nan)
    ok = np.isfinite(p)
    if ok.sum() == 0:
        return q
    pv = p[ok]
    m = len(pv)
    order = np.argsort(pv)
    ranked = pv[order] * m / (np.arange(m) + 1.0)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]     # enforce monotonicity
    out = np.empty(m)
    out[order] = np.clip(ranked, 0.0, 1.0)
    q[ok] = out
    return q


def depth_match_indices(depth_a, depth_b, n_bins=10, seed=0):
    """
    Depth-matching identical in spirit to depth_match_two_groups() in the main notebook:
    joint depth-quantile bins, then down-sample the larger group to the smaller one inside each bin.
    Returns (idx_a, idx_b) = integer positions INTO depth_a and depth_b.
    """
    rng = np.random.default_rng(seed)
    da = np.asarray(depth_a, dtype=float)
    db = np.asarray(depth_b, dtype=float)
    edges = np.quantile(np.concatenate([da, db]), np.linspace(0, 1, n_bins + 1))
    edges[-1] += 1.0
    keep_a, keep_b = [], []
    for i in range(n_bins):
        in_a = np.where((da >= edges[i]) & (da < edges[i + 1]))[0]
        in_b = np.where((db >= edges[i]) & (db < edges[i + 1]))[0]
        m = min(len(in_a), len(in_b))
        if m > 0:
            keep_a.extend(rng.choice(in_a, m, replace=False))
            keep_b.extend(rng.choice(in_b, m, replace=False))
    return np.asarray(keep_a, dtype=int), np.asarray(keep_b, dtype=int)


# ----------------------------------------------------------------------------
# classifier controls (review point 2)
# ----------------------------------------------------------------------------
def _make_clf(scale):
    lr = LogisticRegression(max_iter=1000)
    return make_pipeline(StandardScaler(), lr) if scale else lr


def _cv_accuracy(X, y, seed, n_splits=5, scale=False):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    accs = []
    for tr, te in skf.split(X, y):
        m = _make_clf(scale).fit(X[tr], y[tr])
        accs.append((m.predict(X[te]) == y[te]).mean())
    return float(np.mean(accs)), float(np.std(accs))


def _cv_auc(X, y, seed, n_splits=5, scale=False):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    score = np.zeros(len(y))
    for tr, te in skf.split(X, y):
        m = _make_clf(scale).fit(X[tr], y[tr])
        score[te] = m.predict_proba(X[te])[:, 1]
    return float(roc_auc_score(y, score))


def _one_perm(X, y, seed, n_splits, scale):
    rng = np.random.default_rng(seed)
    yp = rng.permutation(y)
    try:
        return _cv_accuracy(X, yp, seed=int(seed) % (2 ** 31 - 1), n_splits=n_splits, scale=scale)[0]
    except Exception:
        return np.nan


def _perm_p(X, y, observed, n_perm, seed, n_splits=5, scale=False):
    """(r+1)/(M+1) permutation p-value for cross-validated accuracy, labels shuffled n_perm times."""
    seeds = [seed * 1_000_003 + i + 1 for i in range(n_perm)]
    try:
        from joblib import Parallel, delayed
        null = Parallel(n_jobs=-1)(delayed(_one_perm)(X, y, s, n_splits, scale) for s in seeds)
    except Exception:
        null = [_one_perm(X, y, s, n_splits, scale) for s in seeds]
    null = np.asarray(null, dtype=float)
    null = null[np.isfinite(null)]
    return float(_p_plus_one((null >= observed - 1e-12).sum(), len(null))), float(null.mean()) if len(null) else np.nan


def classifier_control_suite(X, obs, group_col, pairs, n_pcs=20, n_perm=1000, seed=0,
                             depth_col="total_counts", n_splits=5, match_depth=True):
    """
    For each (group_a, group_b, label) in `pairs` run, on depth-matched cells:
      * PC classifier      : logistic regression on the first `n_pcs` PCs (as in the main analysis)
      * covariate-only     : logistic regression on log(total counts), log(genes detected), mito fraction
                             (whichever exist in `obs`) -> measures how much of the separation a
                             library-level technical signal alone can reproduce
    plus, on the UNMATCHED cells of the pair, the covariate-only balanced accuracy.
    Permutation p-values use p = (r+1)/(M+1) with `n_perm` label shuffles.
    `X` rows must be in the same order as `obs` rows.
    """
    X = np.asarray(X)
    if X.shape[0] != len(obs):
        raise ValueError(f"X has {X.shape[0]} rows but obs has {len(obs)} rows")
    grp = obs[group_col].astype(str).values
    depth = np.asarray(obs[depth_col], dtype=float) if depth_col in obs.columns else None

    cov_cols = []
    cov_mat = []
    for col, use_log in [("total_counts", True), ("n_genes_detected", True), ("mito_fraction", False)]:
        if col in obs.columns:
            v = np.asarray(obs[col], dtype=float)
            cov_mat.append(np.log1p(v) if use_log else v)
            cov_cols.append(col)
    COV = np.column_stack(cov_mat) if cov_mat else None

    k = min(n_pcs, X.shape[1])
    rows = []
    for (ga, gb, label) in pairs:
        row = {"comparison": label, "group_a": ga, "group_b": gb, "n_pcs": k,
               "n_perm": n_perm, "min_attainable_p": 1.0 / (n_perm + 1)}
        ia_all = np.where(grp == str(ga))[0]
        ib_all = np.where(grp == str(gb))[0]
        row["n_a_all"], row["n_b_all"] = len(ia_all), len(ib_all)
        if min(len(ia_all), len(ib_all)) < 10:
            row["note"] = "skipped: fewer than 10 cells in a group"
            rows.append(row)
            continue

        if match_depth and depth is not None:
            ja, jb = depth_match_indices(depth[ia_all], depth[ib_all], seed=seed)
            sel_a, sel_b = ia_all[ja], ib_all[jb]
        else:
            sel_a, sel_b = ia_all, ib_all
        row["n_a_matched"], row["n_b_matched"] = len(sel_a), len(sel_b)
        if min(len(sel_a), len(sel_b)) < 10:
            row["note"] = "skipped: fewer than 10 cells per group after depth matching"
            rows.append(row)
            continue

        sel = np.concatenate([sel_a, sel_b])
        y = np.concatenate([np.zeros(len(sel_a), int), np.ones(len(sel_b), int)])

        # --- PC classifier
        Xp = X[sel][:, :k]
        acc, acc_sd = _cv_accuracy(Xp, y, seed, n_splits)
        row["acc_pc"], row["acc_pc_sd"] = acc, acc_sd
        row["auc_pc"] = _cv_auc(Xp, y, seed, n_splits)
        row["perm_p_pc"], row["perm_null_mean_pc"] = _perm_p(Xp, y, acc, n_perm, seed, n_splits)

        # --- covariate-only classifier (matched cells)
        if COV is not None:
            Xc = COV[sel]
            cacc, cacc_sd = _cv_accuracy(Xc, y, seed, n_splits, scale=True)
            row["acc_cov_only"], row["acc_cov_only_sd"] = cacc, cacc_sd
            row["auc_cov_only"] = _cv_auc(Xc, y, seed, n_splits, scale=True)
            row["perm_p_cov_only"], _ = _perm_p(Xc, y, cacc, n_perm, seed, n_splits, scale=True)
            # --- covariate-only on all (unmatched) cells of the pair, balanced accuracy
            sel_all = np.concatenate([ia_all, ib_all])
            y_all = np.concatenate([np.zeros(len(ia_all), int), np.ones(len(ib_all), int)])
            skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
            pred = np.zeros(len(y_all), int)
            for tr, te in skf.split(COV[sel_all], y_all):
                pred[te] = _make_clf(True).fit(COV[sel_all][tr], y_all[tr]).predict(COV[sel_all][te])
            row["bal_acc_cov_only_unmatched"] = float(balanced_accuracy_score(y_all, pred))
            row["covariates_used"] = ",".join(cov_cols)
        row["note"] = ""
        rows.append(row)
    return pd.DataFrame(rows)


def pc_count_sensitivity(X, y, pc_counts=(5, 10, 15, 20, 30, 50), n_perm=200, seed=0, n_splits=5):
    """Cross-validated accuracy / AUC / permutation p of the PC classifier as a function of #PCs used."""
    X = np.asarray(X)
    y = np.asarray(y).astype(int)
    rows = []
    for k in pc_counts:
        kk = min(int(k), X.shape[1])
        Xk = X[:, :kk]
        acc, sd = _cv_accuracy(Xk, y, seed, n_splits)
        p, null_mean = _perm_p(Xk, y, acc, n_perm, seed, n_splits)
        rows.append({"n_pcs": kk, "acc": acc, "acc_sd": sd, "auc": _cv_auc(Xk, y, seed, n_splits),
                     "perm_null_mean": null_mean, "perm_p": p, "n_perm": n_perm})
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------
# correlation dimension, done properly (review point 5)
# ----------------------------------------------------------------------------
def _window(n_radii, scaling_frac):
    lo, hi = int(n_radii * scaling_frac[0]), int(n_radii * scaling_frac[1])
    if hi <= lo:
        hi = lo + 2
    return lo, hi


def _corr_sum(dists_sorted, radii):
    # C(r) = fraction of the i<j pairs with distance < r   (i != j, each pair counted once)
    return np.searchsorted(dists_sorted, radii, side="left") / float(len(dists_sorted))


def _slope(dists_sorted, radii, scaling_frac):
    C = _corr_sum(dists_sorted, radii)
    lo, hi = _window(len(radii), scaling_frac)
    r, c = radii[lo:hi], C[lo:hi]
    ok = c > 0
    if ok.sum() < 2:
        return np.nan
    return float(np.polyfit(np.log(r[ok]), np.log(c[ok]), 1)[0])


def _subsample(P, n, rng):
    return P[rng.choice(len(P), size=n, replace=False)]        # WITHOUT replacement -> no duplicate points


def gp_dimension(points, radii=None, scaling_frac=SCALING_FRAC_DEFAULT, n=None, rng=None,
                 n_radii=N_RADII_DEFAULT):
    """
    Grassberger-Procaccia correlation dimension using i != j pairs only (scipy pdist returns each i<j pair
    once) and, if `n` is given, a sub-sample drawn WITHOUT replacement.  If `radii` is None a private
    radius grid (5th-95th percentile of the distances) is used, as in the original estimator.
    """
    P = np.asarray(points, dtype=float)
    rng = rng if rng is not None else np.random.default_rng(0)
    if n is not None:
        P = _subsample(P, n, rng)
    d = np.sort(pdist(P))
    if radii is None:
        dpos = d[d > 0]
        lo_, hi_ = np.percentile(dpos, [5, 95])
        radii = np.logspace(np.log10(lo_), np.log10(hi_), n_radii)
    return _slope(d, np.asarray(radii), scaling_frac)


def common_radii(emb, n, n_radii=N_RADII_DEFAULT, pct=(5, 95), n_rep=5, seed=0):
    """ONE radius grid shared by every condition (pooled 5th-95th percentile of equal-n pairwise distances)."""
    rng = np.random.default_rng(seed)
    pooled = []
    for c, P in emb.items():
        P = np.asarray(P, dtype=float)
        m = min(n, len(P))
        for _ in range(n_rep):
            pooled.append(pdist(_subsample(P, m, rng)))
    d = np.concatenate(pooled)
    d = d[d > 0]
    lo_, hi_ = np.percentile(d, pct)
    return np.logspace(np.log10(lo_), np.log10(hi_), n_radii)


def d2_equal_n(emb, n, n_rep=200, seed=0, n_radii=N_RADII_DEFAULT, scaling_frac=SCALING_FRAC_DEFAULT):
    """
    D2 for every condition at the SAME number of cells `n`, on ONE shared radius grid, i != j, no duplicates.
    Returns (table, draws, radii):  table indexed by condition (mean, sd, 95% interval over `n_rep`
    random equal-n draws), draws = {condition: array of D2 draws}, radii = the shared grid.
    """
    short = {c: len(P) for c, P in emb.items() if len(P) < n}
    if short:
        raise ValueError(f"n={n} exceeds available cells for: {short}")
    radii = common_radii(emb, n, n_radii=n_radii, seed=seed)
    draws, rows = {}, []
    for ci, (c, P) in enumerate(emb.items()):
        rng = np.random.default_rng(seed * 7919 + ci + 1)
        P = np.asarray(P, dtype=float)
        vals = np.array([_slope(np.sort(pdist(_subsample(P, n, rng))), radii, scaling_frac)
                         for _ in range(n_rep)])
        draws[c] = vals
        v = vals[np.isfinite(vals)]
        rows.append({"condition": c, "n_available": len(P), "n_used": n, "n_rep": n_rep,
                     "D2_mean": v.mean(), "D2_sd": v.std(ddof=1) if len(v) > 1 else np.nan,
                     "ci_lo": np.percentile(v, 2.5), "ci_hi": np.percentile(v, 97.5),
                     "note": "uses ALL cells every draw: interval has no sampling spread" if len(P) == n else ""})
    return pd.DataFrame(rows).set_index("condition"), draws, radii


def plot_loglog_correlation_sum(emb, radii, n, seed=0, ax=None, scaling_frac=SCALING_FRAC_DEFAULT):
    """log C(r) vs log r for every condition (one equal-n draw each) with the fitting window shaded."""
    import matplotlib.pyplot as plt
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 5))
    radii = np.asarray(radii)
    lo, hi = _window(len(radii), scaling_frac)
    cmap = plt.get_cmap("tab10")
    for ci, (c, P) in enumerate(emb.items()):
        rng = np.random.default_rng(seed * 7919 + ci + 1)
        d = np.sort(pdist(_subsample(np.asarray(P, float), min(n, len(P)), rng)))
        C = _corr_sum(d, radii)
        ok = C > 0
        ax.plot(np.log(radii[ok]), np.log(C[ok]), "o-", ms=3, lw=1, color=cmap(ci % 10), label=str(c))
    ax.axvspan(np.log(radii[lo]), np.log(radii[hi - 1]), color="grey", alpha=0.15,
               label="fitting window (D2 = slope)")
    ax.set_xlabel("log r")
    ax.set_ylabel("log C(r)")
    ax.set_title(f"Correlation sum, shared radii, n = {n} cells per condition")
    ax.legend(fontsize=7, ncol=2)
    return ax


def d2_difference_test(A, B, n, n_rep=1000, seed=0, n_radii=N_RADII_DEFAULT,
                       scaling_frac=SCALING_FRAC_DEFAULT):
    """
    Direct test of D2(A) - D2(B) at equal n on a shared radius grid.
    * estimate / interval: mean and 2.5-97.5% range of the difference over `n_rep` paired random draws
    * p-value: label-permutation null (pool A and B, re-split at random, same procedure), two-sided,
               p = (r+1)/(M+1)
    """
    A, B = np.asarray(A, float), np.asarray(B, float)
    n_eff = int(min(n, len(A), len(B)))
    radii = common_radii({"A": A, "B": B}, n_eff, n_radii=n_radii, seed=seed)
    rng = np.random.default_rng(seed + 12345)
    diffs = np.empty(n_rep)
    for i in range(n_rep):
        a = _slope(np.sort(pdist(_subsample(A, n_eff, rng))), radii, scaling_frac)
        b = _slope(np.sort(pdist(_subsample(B, n_eff, rng))), radii, scaling_frac)
        diffs[i] = a - b
    d_ok = diffs[np.isfinite(diffs)]
    obs = float(d_ok.mean())

    pooled = np.vstack([A, B])
    n_null = int(min(n_eff, len(pooled) // 2))
    null = np.empty(n_rep)
    for i in range(n_rep):
        perm = rng.permutation(len(pooled))
        a = _slope(np.sort(pdist(pooled[perm[:n_null]])), radii, scaling_frac)
        b = _slope(np.sort(pdist(pooled[perm[n_null:2 * n_null]])), radii, scaling_frac)
        null[i] = a - b
    null = null[np.isfinite(null)]
    p = _p_plus_one((np.abs(null) >= abs(obs)).sum(), len(null))
    return {"n_per_group": n_eff, "n_rep": n_rep,
            "diff_mean": round(obs, 4),
            "diff_lo_2.5": round(float(np.percentile(d_ok, 2.5)), 4),
            "diff_hi_97.5": round(float(np.percentile(d_ok, 97.5)), 4),
            "perm_null_sd": round(float(null.std(ddof=1)), 4),
            "perm_p_two_sided": round(float(p), 4)}


def d2_pc_sensitivity(emb, pc_counts=(10, 15, 20, 30), n=None, n_rep=50, seed=0):
    """D2 at equal n as a function of how many PCs are used. Long table: condition, n_pcs, D2_mean, D2_sd."""
    if n is None:
        n = min(len(P) for P in emb.values())
    rows = []
    for k in pc_counts:
        sub = {c: np.asarray(P)[:, :min(int(k), np.asarray(P).shape[1])] for c, P in emb.items()}
        tab, _, _ = d2_equal_n(sub, n=n, n_rep=n_rep, seed=seed)
        for c in tab.index:
            rows.append({"condition": c, "n_pcs": int(k), "D2_mean": tab.loc[c, "D2_mean"],
                         "D2_sd": tab.loc[c, "D2_sd"], "n_used": n})
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------
# depth controls (review point 8)
# ----------------------------------------------------------------------------
def radial_depth_correlation(pcs, log_depth, n_pcs=2):
    """Spearman correlation between the distance from the origin in the first `n_pcs` PCs and log depth."""
    pcs = np.asarray(pcs, dtype=float)
    radius = np.linalg.norm(pcs[:, :n_pcs], axis=1)
    rho, p = stats.spearmanr(radius, np.asarray(log_depth, dtype=float))
    return float(rho), float(p)


def binomial_thin_counts(X, target, seed=0):
    """
    Binomial thinning of a raw-count matrix: every count of a cell is kept with probability
    target / (cell total), so each retained cell has expected total = `target`.
    Cells with total < target cannot be thinned and are flagged keep = False.
    Returns (X_thinned [same shape, sparse CSR float32], keep [bool array]).
    """
    rng = np.random.default_rng(seed)
    X = sp.csr_matrix(X)
    totals = np.asarray(X.sum(axis=1)).ravel()
    keep = totals >= target
    p_row = np.where(keep, target / np.maximum(totals, 1.0), 1.0)
    p_nnz = np.repeat(p_row, np.diff(X.indptr))
    Xt = X.copy()
    Xt.data = rng.binomial(np.rint(X.data).astype(np.int64), p_nnz).astype(np.float32)
    Xt.eliminate_zeros()
    return Xt, keep


def _scrublet_like(counts, expected_rate=0.06, sim_ratio=2.0, n_pcs=30, hvg_pctl=85, seed=0):
    """
    Simulated-doublet / kNN doublet score in the spirit of Scrublet (Wolock et al. 2019), written with
    scikit-learn only.  Returns (scores, threshold).
    """
    from sklearn.decomposition import TruncatedSVD
    from sklearn.neighbors import NearestNeighbors

    rng = np.random.default_rng(seed)
    C = sp.csr_matrix(counts, dtype=np.float64)
    n = C.shape[0]
    n_sim = int(sim_ratio * n)
    i1 = rng.integers(0, n, n_sim)
    i2 = rng.integers(0, n, n_sim)
    same = i1 == i2
    i2[same] = (i2[same] + 1) % n
    S = C[i1] + C[i2]

    tot_obs = np.asarray(C.sum(1)).ravel()
    target = np.median(tot_obs)
    Cn = sp.diags(target / np.maximum(tot_obs, 1)) @ C
    Sn = sp.diags(target / np.maximum(np.asarray(S.sum(1)).ravel(), 1)) @ S

    # gene filter (expressed in >=3 cells, >=3 counts), then the most variable genes (Fano-like)
    ncell = np.asarray((C > 0).sum(0)).ravel()
    ntot = np.asarray(C.sum(0)).ravel()
    mean = np.asarray(Cn.mean(0)).ravel()
    sq = np.asarray(Cn.multiply(Cn).mean(0)).ravel()
    var = sq - mean ** 2
    fano = var / np.maximum(mean, 1e-12)
    cand = (ncell >= 3) & (ntot >= 3) & (mean > 0)
    cut = np.percentile(fano[cand], hvg_pctl) if cand.sum() else np.inf
    genes = np.where(cand & (fano >= cut))[0]
    if len(genes) < 10:
        genes = np.where(cand)[0]

    def z(M):
        M = np.log1p(M[:, genes].toarray())
        return M
    Zo, Zs = z(Cn), z(Sn)
    mu, sd = Zo.mean(0), Zo.std(0) + 1e-9
    Zo, Zs = (Zo - mu) / sd, (Zs - mu) / sd
    Zo, Zs = np.clip(Zo, -10, 10), np.clip(Zs, -10, 10)
    k_pc = int(min(n_pcs, len(genes) - 1, n - 1))
    svd = TruncatedSVD(n_components=k_pc, random_state=seed).fit(Zo)
    Po, Ps = svd.transform(Zo), svd.transform(Zs)

    comb = np.vstack([Po, Ps])
    k = int(round(0.5 * math.sqrt(n)))
    k = max(k, 5)
    k_adj = int(round(k * (1 + n_sim / n)))
    k_adj = min(k_adj, comb.shape[0] - 1)
    nn = NearestNeighbors(n_neighbors=k_adj + 1).fit(comb)
    _, ind = nn.kneighbors(comb)
    ind = ind[:, 1:]                                   # drop self
    is_sim = np.r_[np.zeros(n, bool), np.ones(n_sim, bool)]
    q = is_sim[ind].mean(1)
    r = n_sim / n
    rho = expected_rate

    def score(qv):
        return (qv * rho / r) / (1 - rho - qv * (1 - rho - rho / r))
    s_all = np.clip(score(q), 0, None)
    obs_scores, sim_scores = s_all[:n], s_all[n:]

    thr = None
    try:
        from skimage.filters import threshold_minimum
        thr = float(threshold_minimum(sim_scores))
    except Exception:
        thr = None
    if thr is None or not np.isfinite(thr) or (obs_scores >= thr).mean() > 0.30 or (obs_scores >= thr).sum() == 0:
        thr = float(np.quantile(obs_scores, 1 - expected_rate))      # fall back to the expected rate
    return obs_scores, thr


def scrublet_by_library(counts, libraries, expected_doublet_rate=0.06, seed=0, prefer_package=True):
    """
    Doublet scores + calls computed SEPARATELY for each library on raw counts.
    Uses the real `scrublet` package when it imports and runs; otherwise uses the built-in
    Scrublet-style implementation above.  The method used for each library is stored in
    `reviewer_controls.LAST_DOUBLET_METHOD` (and printed).
    Returns (scores [float array], calls [bool array]).
    """
    C = sp.csr_matrix(counts)
    libs = np.asarray(libraries).astype(str)
    scores = np.zeros(C.shape[0], dtype=float)
    calls = np.zeros(C.shape[0], dtype=bool)
    LAST_DOUBLET_METHOD.clear()
    pkg = None
    if prefer_package:
        try:
            import scrublet as pkg  # noqa
        except Exception:
            pkg = None
    for lib in [str(l) for l in pd.unique(libs)]:
        idx = np.where(libs == lib)[0]
        sub = C[idx]
        if len(idx) < 50:
            LAST_DOUBLET_METHOD[lib] = "skipped (<50 cells)"
            continue
        done = False
        if pkg is not None:
            try:
                scrub = pkg.Scrublet(sub.tocsc().astype(np.float64), expected_doublet_rate=expected_doublet_rate,
                                     random_state=seed)
                s, c = scrub.scrub_doublets(verbose=False)
                s = np.asarray(s, dtype=float)
                if c is None:
                    thr = float(np.quantile(s, 1 - expected_doublet_rate))
                    c = s >= thr
                scores[idx], calls[idx] = s, np.asarray(c, dtype=bool)
                LAST_DOUBLET_METHOD[lib] = "scrublet package"
                done = True
            except Exception:
                done = False
        if not done:
            s, thr = _scrublet_like(sub, expected_rate=expected_doublet_rate, seed=seed)
            scores[idx], calls[idx] = s, s >= thr
            LAST_DOUBLET_METHOD[lib] = "built-in Scrublet-style (scikit-learn)"
    print("doublet method per library:", dict(LAST_DOUBLET_METHOD))
    return scores, calls


# ----------------------------------------------------------------------------
# 30-bin series, CCM / PE / TE controls (review point 4)
# ----------------------------------------------------------------------------
def order_key_bins(key, n_bins=30):
    """
    Bin the ordering key (day + within-day position) into `n_bins` equal-width bins, exactly as in the
    original analysis.  Returns (bin index per cell [same order as `key`], occupancy per bin, #empty bins).
    """
    key = np.asarray(key, dtype=float)
    edges = np.linspace(key.min(), key.max(), n_bins + 1)
    b_idx = np.clip(np.digitize(key, edges) - 1, 0, n_bins - 1)
    occ = np.bincount(b_idx, minlength=n_bins)
    return b_idx, occ, int((occ == 0).sum())


def binned_means(values, b_idx, n_bins=None, fill="nan"):
    """
    Mean of `values` within each bin.
      fill='drop'   -> occupied bins only (NO interpolation)
      fill='nan'    -> all bins, empty bins = NaN
      fill='interp' -> all bins, empty bins linearly interpolated (what the original analysis did)
    """
    values = np.asarray(values, dtype=float)
    b_idx = np.asarray(b_idx)
    if n_bins is None:
        n_bins = int(b_idx.max()) + 1
    sums = np.bincount(b_idx, weights=values, minlength=n_bins)
    cnt = np.bincount(b_idx, minlength=n_bins)
    out = np.full(n_bins, np.nan)
    occ = cnt > 0
    out[occ] = sums[occ] / cnt[occ]
    if fill == "drop":
        return out[occ]
    if fill == "interp":
        return pd.Series(out).interpolate(limit_direction="both").values
    return out


def ccm_settings_report(n_points, embed_dim=2, lag=1, train_percent=0.75):
    """The exact CCM settings implied by skccm for a series of `n_points` (for the Methods section)."""
    n_vec = n_points - lag * (embed_dim - 1)
    n_train = int(n_vec * train_percent)
    n_test = n_vec - n_train
    step = max(1, (n_train - 8) // 6)
    libs = list(range(8, n_train, step))
    return {"series_length": n_points, "embedding_dim": embed_dim, "lag": lag,
            "embedded_vectors": n_vec, "train_vectors": n_train, "test_vectors": n_test,
            "library_lengths": libs, "train_percent": train_percent}


def iaaft(x, rng, n_iter=100):
    """Iterative amplitude-adjusted Fourier-transform surrogate (keeps the value distribution AND the spectrum)."""
    x = np.asarray(x, dtype=float)
    n = len(x)
    xs = np.sort(x)
    amp = np.abs(np.fft.rfft(x))
    s = rng.permutation(x)
    for _ in range(n_iter):
        phases = np.angle(np.fft.rfft(s))
        s_new = np.fft.irfft(amp * np.exp(1j * phases), n=n)
        s_new = xs[np.argsort(np.argsort(s_new))]          # rank-remap to the original values
        if np.array_equal(s_new, s):
            break
        s = s_new
    return s


def twin_surrogate(x, rng, dim=2, lag=1, rec_rate=0.10):
    """
    Twin surrogate (Thiel et al. 2006): random walk through the delay-embedded state space that jumps
    between 'twin' states (states with identical recurrence neighbourhoods).  Preserves the deterministic
    structure while breaking the specific temporal alignment with the other series.
    """
    x = np.asarray(x, dtype=float)
    n = len(x)
    N = n - (dim - 1) * lag
    if N < 6:
        return rng.permutation(x)
    E = np.column_stack([x[i * lag:i * lag + N] for i in range(dim)])
    D = squareform(pdist(E, metric="chebyshev"))
    eps = np.quantile(D[np.triu_indices(N, 1)], rec_rate)
    R = D <= eps
    groups = {}
    for i in range(N):
        groups.setdefault(R[i].tobytes(), []).append(i)
    twins = [groups[R[i].tobytes()] for i in range(N)]
    path = [int(rng.integers(N))]
    for _ in range(N - 1):
        j = path[-1]
        t = twins[j][int(rng.integers(len(twins[j])))]
        nxt = t + 1
        if nxt >= N:
            nxt = int(rng.integers(N))
        path.append(nxt)
    s = x[np.array(path)]
    tail = [x[min(path[-1] + k, n - 1)] for k in range(1, n - N + 1)]
    return np.concatenate([s, tail]) if tail else s


def matched_random_genes(gene_mean, gene_detect, target_idx, exclude, n_genes=300, tol=0.25, seed=0):
    """
    Indices of genes whose mean expression and detection fraction are both within +/- `tol` (relative) of
    the target gene's.  If fewer than `n_genes` qualify the tolerance is widened stepwise; if still short
    the closest genes are taken.  Target and `exclude` genes are never returned.
    """
    rng = np.random.default_rng(seed)
    gm, gd = np.asarray(gene_mean, float), np.asarray(gene_detect, float)
    banned = np.zeros(len(gm), bool)
    banned[list(exclude)] = True
    banned[target_idx] = True
    m0, d0 = gm[target_idx], gd[target_idx]
    t = tol
    cand = np.array([], dtype=int)
    while t <= 2.0:
        ok = (~banned) & (np.abs(gm - m0) <= t * m0) & (np.abs(gd - d0) <= t * d0)
        cand = np.where(ok)[0]
        if len(cand) >= n_genes:
            break
        t *= 1.5
    if len(cand) < n_genes:
        dist = np.abs(np.log((gm + 1e-9) / (m0 + 1e-9))) + np.abs(np.log((gd + 1e-9) / (d0 + 1e-9)))
        dist[banned] = np.inf
        cand = np.argsort(dist)[:n_genes]
        cand = cand[np.isfinite(dist[cand])]
    if len(cand) > n_genes:
        cand = rng.choice(cand, n_genes, replace=False)
    return np.asarray(cand, dtype=int)


def random_gene_ccm_control(observed, null_skills):
    """Where does the observed CCM skill sit among the skills of expression-matched random genes?"""
    v = np.asarray(null_skills, dtype=float)
    v = v[np.isfinite(v)]
    if len(v) == 0:
        return {"observed": float(observed), "n_null_genes": 0}
    return {"observed": float(observed), "null_mean": float(v.mean()), "null_sd": float(v.std(ddof=1)),
            "null_q95": float(np.quantile(v, 0.95)), "null_max": float(v.max()),
            "percentile_of_observed": float((v < observed).mean() * 100),
            "p_one_sided": float(_p_plus_one((v >= observed).sum(), len(v))),
            "n_null_genes": int(len(v))}


In [ ]:
# E1 -- import the helper module and set the p-value convention used everywhere
import sys, os, importlib
sys.path.insert(0, '/content')
import reviewer_controls as rc; importlib.reload(rc)
import numpy as np, pandas as pd
OUT_DIR = '/content/section_E_outputs'      # every table / figure of Section E is saved here
os.makedirs(OUT_DIR, exist_ok=True); os.chdir(OUT_DIR)
N_PERM = 1000          # 1,000 permutations -> smallest attainable p = 1/1001 ~ 0.001
def p_plus_one(p_old, M_old=200):
    # convert an old 'fraction of surrogates >= real' p (r/M) to the (r+1)/(M+1) convention
    return (np.asarray(p_old) * M_old + 1) / (M_old + 1)
obs_all = adata_qc2.obs.loc[adata_hvg2.obs_names].copy()          # total_counts, n_genes_detected, sample_label ...
obs_all['grp'] = obs_all['sample_label'].astype(str)                # keeps the two Day-11 libraries apart
PCS = adata_hvg2.obsm['X_pca']

In [ ]:
# E2 -- review point 2: batch-confounding controls (each condition is ONE library)
pairs = [
    ('D11', 'D11_rep2', 'Day 11 library A vs library B (same condition, different library)'),
    ('D0',  'D19_DMSO_holiday', 'Day 0 vs Day 19 withdrawn'),
    ('D0',  'D19_ERL_retreated', 'Day 0 vs Day 19 re-treated'),
    ('D19_DMSO_holiday', 'D19_ERL_retreated', 'Day 19 withdrawn vs Day 19 re-treated (main comparison)'),
]
table_E = rc.classifier_control_suite(PCS, obs_all, 'grp', pairs, n_pcs=20, n_perm=N_PERM, seed=0)
print(table_E.round(3).to_string())
table_E.to_csv('table_E_controls.csv', index=False)
print("\nHow to read it: if the Day-11 library-vs-library accuracy (or the covariate-only accuracy) is close to the "
      "88.2% obtained for withdrawn vs re-treated, that accuracy cannot be attributed to biology.")

In [ ]:
# E3 -- PC-count sensitivity of the classifier (why 20 PCs?)
ia, ib = rc.depth_match_indices(obs_all.loc[obs_all.grp=='D19_DMSO_holiday','total_counts'].values,
                                obs_all.loc[obs_all.grp=='D19_ERL_retreated','total_counts'].values)
sel_a = np.where(obs_all.grp=='D19_DMSO_holiday')[0][ia]; sel_b = np.where(obs_all.grp=='D19_ERL_retreated')[0][ib]
sel = np.r_[sel_a, sel_b]; y = np.r_[np.zeros(len(sel_a),int), np.ones(len(sel_b),int)]
sens = rc.pc_count_sensitivity(PCS[sel], y, pc_counts=(5,10,15,20,30,50), n_perm=200)
print(sens.round(3)); sens.to_csv('table_E_pc_sensitivity_classifier.csv', index=False)

In [ ]:
# E4 -- review point 5: correlation dimension redone properly
#  (a) i != j pairs only, sampling WITHOUT replacement (no duplicate points)      -> rc.gp_dimension
#  (b) every condition at the SAME n                                               -> rc.d2_equal_n
#  (c) one radius grid shared by all conditions                                    -> rc.common_radii
#  (d) log C(r) vs log r plot with the fitting window                              -> rc.plot_loglog_correlation_sum
#  (e) direct test of D2(withdrawn) - D2(re-treated)                               -> rc.d2_difference_test
#  (f) PC-count sensitivity (why 15 PCs?)                                          -> rc.d2_pc_sensitivity
emb = {c: adata_hvg2.obsm['X_pca'][(adata_hvg2.obs['condition']==c).values] for c in MAIN_ARM}
# for the reversibility pair use the depth-matched cells already selected in adata_reversibility
for c in ['D19_DMSO_holiday','D19_ERL_retreated']:
    emb[c] = adata_reversibility.obsm['X_pca'][(adata_reversibility.obs['condition']==c).values]
emb15 = {c: P[:, :15] for c, P in emb.items()}
n_equal = min(len(P) for P in emb15.values())
print("equal n =", n_equal, "(smallest condition)")
tab_eq, draws_eq, radii = rc.d2_equal_n(emb15, n=n_equal, n_rep=200, seed=0)
print(tab_eq.round(3)); tab_eq.to_csv('table_E_D2_equal_n.csv')
import matplotlib.pyplot as plt
ax = rc.plot_loglog_correlation_sum(emb15, radii, n_equal); plt.tight_layout()
plt.savefig('figureS_D2_loglog.png', dpi=300)
diff = rc.d2_difference_test(emb15['D19_DMSO_holiday'], emb15['D19_ERL_retreated'], n=n_equal, n_rep=1000)
print("D2(withdrawn) - D2(re-treated):", diff)
sens_d2 = rc.d2_pc_sensitivity({c: emb[c] for c in MAIN_ARM+['D19_DMSO_holiday','D19_ERL_retreated']},
                               pc_counts=(10,15,20,30), n=n_equal, n_rep=50)
sens_d2.to_csv('table_E_D2_pc_sensitivity.csv'); print(sens_d2.pivot(index='condition', columns='n_pcs', values='D2_mean').round(3))

In [ ]:
# E5 -- review point 8: depth. (1) radial depth effect, (2) count thinning, (3) Scrublet doublets
rho, p = rc.radial_depth_correlation(PCS, np.log(obs_all['total_counts'].values), n_pcs=2)
print(f"Spearman(radius in PC1-PC2, log depth) = {rho:.3f} (p={p:.2g})   <-- residual radial depth effect")

# (2) binomial thinning to a common depth, then re-run classifier + PCA pipeline on thinned counts
import scanpy as sc
TARGET = 600
Xraw = adata_qc2.layers['raw_counts']
Xthin, keep = rc.binomial_thin_counts(Xraw, TARGET, seed=0)
ad_thin = adata_qc2[keep].copy(); ad_thin.X = Xthin[keep].astype(np.float32)
sc.pp.normalize_total(ad_thin, target_sum=1e4); sc.pp.log1p(ad_thin)
sc.pp.highly_variable_genes(ad_thin, n_top_genes=2000, flavor='seurat')
ad_thin = ad_thin[:, ad_thin.var.highly_variable | ad_thin.var_names.isin(dt_markers)].copy()
sc.pp.scale(ad_thin, max_value=10); sc.tl.pca(ad_thin, n_comps=50)
ad_thin.uns.pop('log1p', None)   # X now holds raw (thinned) counts again; avoid scanpy's 'already log-transformed' note
obs_thin = ad_thin.obs.copy(); obs_thin['grp'] = obs_thin['sample_label'].astype(str)
# recompute the covariates FROM THE THINNED COUNTS (the copied obs columns still describe the un-thinned cells)
_Xk = Xthin[keep]
obs_thin['total_counts'] = np.asarray(_Xk.sum(1)).ravel()            # ~TARGET for every cell (binomial thinning), unrelated to library
obs_thin['n_genes_detected'] = np.asarray((_Xk > 0).sum(1)).ravel()
rho_t, p_t = rc.radial_depth_correlation(ad_thin.obsm['X_pca'], np.log(adata_qc2.obs.loc[ad_thin.obs_names,'total_counts'].values))
print(f"after thinning to {TARGET} counts: radial correlation with ORIGINAL depth = {rho_t:.3f}; cells kept = {keep.sum()}")
thin_tab = rc.classifier_control_suite(ad_thin.obsm['X_pca'], obs_thin, 'grp', pairs, n_pcs=20, n_perm=200)
print(thin_tab.round(3).to_string()); thin_tab.to_csv('table_E_thinned_controls.csv', index=False)

# (3) Scrublet, run per library on raw counts; drop predicted doublets and repeat the main classifier
scores, calls = rc.scrublet_by_library(adata_qc2.layers['raw_counts'], adata_qc2.obs['sample_label'].astype(str).values)
adata_qc2.obs['doublet_score'] = scores; adata_qc2.obs['predicted_doublet'] = calls
print(adata_qc2.obs.groupby('sample_label')['predicted_doublet'].mean().round(3))
ok = ~adata_qc2.obs.loc[adata_hvg2.obs_names,'predicted_doublet'].values
nodbl = rc.classifier_control_suite(PCS[ok], obs_all[ok], 'grp', pairs, n_pcs=20, n_perm=200)
print(nodbl.round(3).to_string()); nodbl.to_csv('table_E_no_doublets_controls.csv', index=False)

In [ ]:
# E6 -- review point 4: CCM / permutation-entropy / transfer-entropy on the 30-bin series
# (a) how many of the 30 bins are EMPTY and were filled by linear interpolation in the original analysis?
key_main = (real_day + (within_day_dist - within_day_dist.min())/(within_day_dist.max()-within_day_dist.min()+1e-9))
b_idx, occ, n_empty = rc.order_key_bins(key_main, n_bins=30)
print("bin occupancy:", occ.tolist()); print(f"EMPTY (interpolated) bins: {n_empty} of 30   <-- quote in Methods")
print("CCM settings:", rc.ccm_settings_report(30, embed_dim=2, lag=1, train_percent=0.75))

# (b) rerun PE / TE / CCM on OCCUPIED bins only (no interpolation)
axis_vals = dist_clean_series.loc[main_arm_names].values
drop_df = pd.DataFrame({'axis': rc.binned_means(axis_vals, b_idx, fill='drop')})
for g in dt_markers:
    gi = np.where(adata_qc2.var_names==g)[0][0]
    e = np.asarray(adata_qc2[main_arm_names, gi].X.todense()).ravel()
    drop_df[g] = rc.binned_means(e, b_idx, fill='drop')  # same occupied bins because bins are defined by cells
print("no-interpolation series length:", len(drop_df))
pe_drop = {name: pe_surrogate_test(drop_df[name].values, m=3, tau=1, n_surrogates=1000) for name in drop_df.columns}
pe_drop_tab = pd.DataFrame(pe_drop, index=['PE','surr_mean','surr_sd','p_old_r/M']).T
pe_drop_tab['p_(r+1)/(M+1)'] = p_plus_one(pe_drop_tab['p_old_r/M'].values, M_old=1000)
print(pe_drop_tab.round(3)); pe_drop_tab.to_csv('table_E_PE_no_interpolation.csv')

# (c) CCM significance against stronger nulls (phase-randomised is already in C0b; add IAAFT and twin surrogates)
def ccm_skill(x, y):
    a, b = run_ccm_pair(x, y); return np.nanmax(a), np.nanmax(b)
def ccm_surrogate_p(x, y, kind, M=200, seed=0):
    rng = np.random.default_rng(seed); real_xy, real_yx = ccm_skill(x, y); s_xy, s_yx = [], []
    for _ in range(M):
        ys = {'iaaft': lambda: rc.iaaft(y, rng), 'twin': lambda: rc.twin_surrogate(y, rng)}[kind]()
        a, b = ccm_skill(x, ys); s_xy.append(a); s_yx.append(b)
    p = lambda s, r: (np.sum(np.array(s) >= r) + 1) / (len(s) + 1)
    return real_xy, p(s_xy, real_xy), real_yx, p(s_yx, real_yx)
rows = []
for g in dt_markers:
    for kind in ['iaaft', 'twin']:
        r1, p1, r2, p2 = ccm_surrogate_p(binned_df[g].values, binned_df['axis'].values, kind)
        rows.append({'gene': g, 'null': kind, 'g2a_rho': r1, 'g2a_p': p1, 'a2g_rho': r2, 'a2g_p': p2})
ccm_strong = pd.DataFrame(rows); print(ccm_strong.round(3).to_string()); ccm_strong.to_csv('table_E_ccm_iaaft_twin.csv', index=False)

# (d) matched random-gene control: is EGFR / SOX4 skill unusual among genes of similar expression?
Xmain = adata_qc2[main_arm_names].X.tocsc()
gene_mean = np.asarray(Xmain.mean(0)).ravel(); gene_detect = np.asarray((Xmain>0).mean(0)).ravel()
exclude = [np.where(adata_qc2.var_names==g)[0][0] for g in dt_markers]
ctrl = {}
for target in ['EGFR', 'SOX4']:
    ti = np.where(adata_qc2.var_names==target)[0][0]
    rand_idx = rc.matched_random_genes(gene_mean, gene_detect, ti, exclude, n_genes=300, tol=0.25, seed=0)
    skills = []
    for gi in rand_idx:
        e = np.asarray(Xmain[:, gi].todense()).ravel()
        series = pd.Series([e[b_idx==b].mean() if (b_idx==b).any() else np.nan for b in range(30)]).interpolate(limit_direction='both').values
        skills.append(ccm_skill(series, binned_df['axis'].values)[0])
    ctrl[target] = rc.random_gene_ccm_control(table2.loc[target,'gene_to_axis_rho'], skills)
    print(target, ctrl[target])
pd.DataFrame(ctrl).T.to_csv('table_E_random_gene_control.csv')

In [ ]:
# E7 -- review point 6: BH q-values using p = (r+1)/(M+1); Table 5 gets q-values
def table_with_q(df, pcols):
    out = df.copy(); allp = np.concatenate([out[c].values for c in pcols]); q = rc.benjamini_hochberg(allp)
    k = 0
    for c in pcols:
        out['q_'+c] = q[k:k+len(out)]; k += len(out)
    return out
t4 = table_with_q(table2.assign(gene_to_axis_p=p_plus_one(table2.gene_to_axis_p), axis_to_gene_p=p_plus_one(table2.axis_to_gene_p)),
                  ['gene_to_axis_p', 'axis_to_gene_p'])
print(t4.round(3)); t4.to_csv('table_E_ccm_qvalues.csv')
# Table 5: q across the 9 gene->trajectory tests, separately for each QC threshold
for name, res in [('original', results_original), ('alternative', results_alternative)]:
    pv = pd.Series({g: v['gene_to_axis_p'] for g, v in res['ccm_results'].items()})
    q = pd.Series(rc.benjamini_hochberg(p_plus_one(pv.values)), index=pv.index)
    print(name); print(pd.DataFrame({'p': p_plus_one(pv.values), 'q': q}).round(3))

In [ ]:
# E8 -- review point 6: verify that the 'alternative' QC run is a genuinely different run
cells_orig = set(results_original['condition_counts'].index)   # (labels only) -- compare counts below
print("cells per condition, original vs alternative:")
print(pd.DataFrame({'original': results_original['condition_counts'], 'alternative': results_alternative['condition_counts']}))
print("total:", results_original['n_total_qc'], "vs", results_alternative['n_total_qc'])
# both runs above used seed=0 -> identical bootstrap / matching random streams; near-identical D2 on ~90% shared cells is therefore expected.
# Re-run BOTH thresholds with independent seeds so the comparison is not seed-locked:
res_o1 = run_downstream_pipeline(adata_raw, 300, 30, 0.30, DT_MARKERS_WANTED, CONDITION_ORDER, MAIN_ARM, DAY_MAP, seed=1)
res_a2 = run_downstream_pipeline(adata_raw, 250, 25, 0.35, DT_MARKERS_WANTED, CONDITION_ORDER, MAIN_ARM, DAY_MAP, seed=2)
for lab, r in [('original seed0', results_original), ('original seed1', res_o1), ('alternative seed0', results_alternative), ('alternative seed2', res_a2)]:
    print(lab, {k: round(v[1], 3) for k, v in r['corr_dim'].items()})

## E9 — save everything from Section E and download it

In [ ]:
import shutil, glob
zip_path = shutil.make_archive(OUT_DIR, 'zip', OUT_DIR)
print("Zipped:", zip_path)
for f in sorted(os.listdir(OUT_DIR)):
    print("  ", f)
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("(not running in Colab, or download blocked -- grab the zip from the path above)", e)